In [1]:

import builtins
import os
import sys
from pathlib import Path
import pandas as _winner_pd

_WINNER_SOURCE_DIR = Path('/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/raw_behavior_by_winner')
_WINNER_RESULTS_DIR = Path('/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3')
_WINNER_CSV = Path('/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/all_subjects_final_data.csv')
if str(_WINNER_SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(_WINNER_SOURCE_DIR))
os.chdir(_WINNER_RESULTS_DIR)

def _winner_input(prompt=''):
    prompt_lower = str(prompt).lower()
    if 'csv' in prompt_lower:
        return str(_WINNER_CSV)
    if 'number of bins' in prompt_lower:
        return '4'
    if 'directory to save' in prompt_lower:
        return str(_WINNER_RESULTS_DIR)
    raise RuntimeError(f'Unexpected interactive prompt: {prompt}')

builtins.input = _winner_input
_winner_df = _winner_pd.read_csv(_WINNER_CSV)
print(
    'MODEL-WINNER INPUT:',
    len(_winner_df), 'trials,',
    _winner_df['session_uid'].nunique(), 'sessions, subjects',
    sorted(_winner_df['subject'].unique()),
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as _winner_plt
_winner_plt.show = lambda *args, **kwargs: _winner_plt.close('all')


MODEL-WINNER INPUT: 291 trials, 1 sessions, subjects ['MC06']


# licking analysis

mean and sem analysis for all trials

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["text.usetex"] = False
plt.rcParams["savefig.dpi"] = 600

DATA_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/all_subjects_final_data.csv"
OUTPUT_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/licking_analysis_figure.pdf"

SESSION_COL = "session_idx"
LATENCY_COL = "lick_reaction_time"
COUNT_COL = "all_lick_count"
OUTCOME_LEVELS = ["Reward", "Punish"]
RNG_SEED = 7


def prepare_licking_dataframe(path, exclude_warmup=True):
    """
    Load the licking CSV and add analysis labels.

    trial_type:
        1 = Short trial
        2 = Long trial

    block_type:
        0 = Neutral block
        1 = Short block
        2 = Long block

    Rarity:
        Short trials are common in Short blocks.
        Long trials are common in Long blocks.
    """
    required = {
        SESSION_COL, "trial_type", "trial_isi", "block_type", "outcome",
        "warm_up", LATENCY_COL, COUNT_COL,
    }

    df = pd.read_csv(path).copy()
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    if exclude_warmup:
        df = df[df["warm_up"].eq(0)].copy()

    df["trial_name"] = df["trial_type"].map({1: "Short", 2: "Long"})
    df["block_name"] = df["block_type"].map({
        0: "Neutral",
        1: "Short Block",
        2: "Long Block",
    })

    df["rarity"] = np.select(
        [
            df["block_type"].eq(1) & df["trial_type"].eq(1),
            df["block_type"].eq(1) & df["trial_type"].eq(2),
            df["block_type"].eq(2) & df["trial_type"].eq(2),
            df["block_type"].eq(2) & df["trial_type"].eq(1),
            df["block_type"].eq(0),
        ],
        ["Common", "Rare", "Common", "Rare", "Neutral"],
        default="Unknown",
    )

    df["outcome_strict"] = df["outcome"].replace({
        "RewardNaive": "Reward",
        "PunishNaive": "Punish",
    })

    conditions = [
        df["block_type"].eq(1) & df["trial_type"].eq(1),
        df["block_type"].eq(1) & df["trial_type"].eq(2),
        df["block_type"].eq(2) & df["trial_type"].eq(2),
        df["block_type"].eq(2) & df["trial_type"].eq(1),
        df["block_type"].eq(0) & df["trial_type"].eq(1),
        df["block_type"].eq(0) & df["trial_type"].eq(2),
    ]

    df["col2_x"] = np.select(
        conditions,
        ["SB\nCom", "SB\nRare", "LB\nCom", "LB\nRare", "NB\nShort", "NB\nLong"],
        default="Unknown",
    )

    df["col4_x"] = np.select(
        conditions[:4],
        ["SB Common", "SB Rare", "LB Common", "LB Rare"],
        default="Exclude",
    )

    if df[["trial_name", "block_name"]].isna().any().any() or (df["rarity"] == "Unknown").any():
        bad = df.loc[
            df["trial_name"].isna() | df["block_name"].isna() | df["rarity"].eq("Unknown"),
            ["trial_type", "block_type"],
        ].drop_duplicates()
        raise ValueError(f"Unexpected trial/block coding found:\n{bad}")

    return df


def format_p(p):
    """Format p-values for compact figure text."""
    if pd.isna(p):
        return "NA"
    return f"{p:.1e}" if p < 0.001 else f"{p:.3f}"


def p_to_stars(p):
    """Convert p-value to significance stars."""
    if pd.isna(p):
        return "N/A"
    if p < 0.0001:
        return "****"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def safe_shapiro_p(values):
    """Run Shapiro-Wilk normality test on paired differences."""
    values = pd.Series(values).dropna().astype(float)
    if len(values) < 3 or values.nunique() < 2:
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(stats.shapiro(values).pvalue)


def paired_session_test(df_subset, mask1, mask2, value_col, label1, label2):
    """
    Compare two conditions using paired session-level means.

    This avoids treating thousands of trials as independent samples.
    If paired session differences look normal, use paired t-test.
    Otherwise, use Wilcoxon signed-rank test.
    """
    df1 = df_subset.loc[mask1, [SESSION_COL, value_col]].dropna(subset=[value_col])
    df2 = df_subset.loc[mask2, [SESSION_COL, value_col]].dropna(subset=[value_col])

    s1 = df1.groupby(SESSION_COL, observed=True)[value_col].mean().rename(label1)
    s2 = df2.groupby(SESSION_COL, observed=True)[value_col].mean().rename(label2)

    paired = pd.concat([s1, s2], axis=1, join="inner").dropna()
    diff = paired[label1] - paired[label2]

    result = {
        "comparison": f"{label1} vs {label2}",
        "value_col": value_col,
        "test": "N/A",
        "statistic": np.nan,
        "p_raw": np.nan,
        "p_holm": np.nan,
        "n_trials": int(len(df1) + len(df2)),
        "n_sessions": int(len(paired)),
        "mean_diff": np.nan,
        "effect_dz": np.nan,
        "normality_p": np.nan,
    }

    if len(paired) < 2:
        return result

    result["mean_diff"] = float(diff.mean())
    result["normality_p"] = safe_shapiro_p(diff)

    sd_diff = float(diff.std(ddof=1))
    if sd_diff > 0:
        result["effect_dz"] = float(diff.mean() / sd_diff)

    if np.allclose(diff.to_numpy(), 0):
        result.update({"test": "no difference", "statistic": 0.0, "p_raw": 1.0})
        return result

    use_paired_t = pd.notna(result["normality_p"]) and result["normality_p"] >= 0.05

    if use_paired_t:
        statistic, p_raw = stats.ttest_rel(paired[label1], paired[label2])
        test_name = "paired t"
    else:
        statistic, p_raw = stats.wilcoxon(
            paired[label1],
            paired[label2],
            zero_method="wilcox",
            alternative="two-sided",
            method="auto",
        )
        test_name = "Wilcoxon"

    result.update({
        "test": test_name,
        "statistic": float(statistic),
        "p_raw": float(p_raw),
    })

    return result


def holm_adjust(p_values):
    """Holm-Bonferroni correction for multiple comparisons."""
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan)

    valid = np.flatnonzero(~np.isnan(p_values))
    if len(valid) == 0:
        return adjusted

    ordered = valid[np.argsort(p_values[valid])]
    running_max = 0.0
    m = len(valid)

    for rank, idx in enumerate(ordered):
        adjusted_p = (m - rank) * p_values[idx]
        running_max = max(running_max, adjusted_p)
        adjusted[idx] = min(running_max, 1.0)

    return adjusted


def add_holm_correction(results):
    """Apply Holm correction separately for latency and lick-count families."""
    for value_col in sorted({r["value_col"] for r in results}):
        idx = [i for i, r in enumerate(results) if r["value_col"] == value_col]
        adjusted = holm_adjust([results[i]["p_raw"] for i in idx])

        for i, p_adj in zip(idx, adjusted):
            results[i]["p_holm"] = float(p_adj) if not pd.isna(p_adj) else np.nan

    return results


def add_result(results, panel, section, df_subset, mask1, mask2, value_col, label1, label2):
    """Run one comparison and store metadata for figure text."""
    result = paired_session_test(df_subset, mask1, mask2, value_col, label1, label2)
    result["panel"] = panel
    result["section"] = section
    result["order"] = len(results)
    results.append(result)


def collect_statistics(df):
    """Collect all pairwise tests shown in the figure."""
    results = []

    df_c1 = df[df["block_type"].ne(0)].copy()
    df_c3 = df_c1[df_c1["outcome_strict"].isin(OUTCOME_LEVELS)].copy()
    df_c4 = df[df["col4_x"].ne("Exclude") & df["outcome_strict"].isin(OUTCOME_LEVELS)].copy()

    for value_col, section in [(LATENCY_COL, "LATENCY"), (COUNT_COL, "COUNTS")]:
        add_result(
            results, "col1", section, df_c1,
            df_c1["trial_name"].eq("Short"),
            df_c1["trial_name"].eq("Long"),
            value_col, "Short", "Long",
        )

        for a, b, la, lb in [
            ("SB\nCom", "SB\nRare", "SB Com", "SB Rare"),
            ("LB\nCom", "LB\nRare", "LB Com", "LB Rare"),
            ("NB\nShort", "NB\nLong", "NB Short", "NB Long"),
        ]:
            add_result(
                results, "col2", section, df,
                df["col2_x"].eq(a),
                df["col2_x"].eq(b),
                value_col, la, lb,
            )

        for trial_name in ["Short", "Long"]:
            subset = df_c3[df_c3["trial_name"].eq(trial_name)].copy()
            add_result(
                results, "col3", section, subset,
                subset["outcome_strict"].eq("Reward"),
                subset["outcome_strict"].eq("Punish"),
                value_col, f"{trial_name}: Rew", "Pun",
            )

        for cat in ["SB Common", "SB Rare", "LB Common", "LB Rare"]:
            subset = df_c4[df_c4["col4_x"].eq(cat)].copy()
            short_label = cat.replace(" Common", " C").replace(" Rare", " R")
            add_result(
                results, "col4", section, subset,
                subset["outcome_strict"].eq("Reward"),
                subset["outcome_strict"].eq("Punish"),
                value_col, f"{short_label}: Rew", "Pun",
            )

    return add_holm_correction(results)


def stat_line(result):
    """Format one statistical result for the figure."""
    if pd.isna(result["p_raw"]):
        return f"{result['comparison']}: N/A, {result['n_trials']}tr/{result['n_sessions']}sess"

    return (
        f"{result['comparison']}: {result['test']}, "
        f"pH={format_p(result['p_holm'])} ({p_to_stars(result['p_holm'])}), "
        f"{result['n_trials']}tr/{result['n_sessions']}sess"
    )


def stats_box_text(results, panel):
    """Build the text shown in the bottom row of each column."""
    lines = []

    for section in ["LATENCY", "COUNTS"]:
        lines.append(f"--- {section} ---")
        section_results = [
            r for r in results
            if r["panel"] == panel and r["section"] == section
        ]
        section_results = sorted(section_results, key=lambda r: r["order"])
        lines.extend(stat_line(r) for r in section_results)
        lines.append("")

    return "\n".join(lines).strip()


def session_mean_table(data, group_cols, value_col):
    """Average trials within each session and condition before plotting/testing."""
    cols = [SESSION_COL] + group_cols + [value_col]

    return (
        data[cols]
        .dropna(subset=group_cols + [value_col])
        .groupby([SESSION_COL] + group_cols, observed=True, as_index=False)[value_col]
        .mean()
    )


def sem(values):
    """Standard error of the session means."""
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        return np.nan
    return float(stats.sem(values, nan_policy="omit"))


def clean_axis(ax):
    """Remove top/right spines and keep the plot background grid-free."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)
    ax.set_axisbelow(True)


def plot_session_dots(
    ax,
    data,
    x_col,
    y_col,
    order,
    hue_col=None,
    hue_order=None,
    title="",
    ylabel="",
    rotation=0,
    separators=None,
    rng=None,
):
    """
    Plot only session-level mean +/- SEM.

    Individual session dots are omitted; the plotted unit remains the
    session-level mean used for the statistics.
    """
    if rng is None:
        rng = np.random.default_rng(RNG_SEED)

    group_cols = [x_col] if hue_col is None else [x_col, hue_col]
    summary = session_mean_table(data, group_cols, y_col)
    base_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    if hue_col is None:
        for x_idx, category in enumerate(order):
            values = summary.loc[summary[x_col].eq(category), y_col].dropna().to_numpy(float)
            if len(values) == 0:
                continue

            ax.errorbar(
                x_idx,
                values.mean(),
                yerr=sem(values),
                fmt="o",
                color="black",
                ecolor="black",
                capsize=4,
                markersize=7,
                zorder=5,
            )

    else:
        if hue_order is None:
            hue_order = list(pd.unique(summary[hue_col].dropna()))

        offsets = np.linspace(-0.18, 0.18, len(hue_order)) if len(hue_order) > 1 else [0.0]
        colors = {level: base_colors[i % len(base_colors)] for i, level in enumerate(hue_order)}

        for h_idx, hue_level in enumerate(hue_order):
            for x_idx, category in enumerate(order):
                values = summary.loc[
                    summary[x_col].eq(category) & summary[hue_col].eq(hue_level),
                    y_col,
                ].dropna().to_numpy(float)

                if len(values) == 0:
                    continue

                center = x_idx + offsets[h_idx]
                label = hue_level if x_idx == 0 else None

                ax.errorbar(
                    center,
                    values.mean(),
                    yerr=sem(values),
                    fmt="o",
                    color=colors[hue_level],
                    ecolor=colors[hue_level],
                    capsize=4,
                    markersize=7,
                    zorder=5,
                    label=label,
                )

        ax.legend(frameon=False, title=None)

    if separators:
        for x in separators:
            ax.axvline(x, color="0.25", linestyle="--", linewidth=1, alpha=0.5)

    ax.set_title(title, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=rotation, ha="right" if rotation else "center")

    clean_axis(ax)


def add_stats_box(ax, text, fontsize=10):
    """Add formatted statistics text to an axis."""
    ax.axis("off")
    ax.text(
        0.5,
        0.5,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        transform=ax.transAxes,
        bbox=dict(
            facecolor="whitesmoke",
            alpha=0.9,
            boxstyle="round,pad=0.8",
            edgecolor="0.85",
        ),
    )


def make_licking_analysis_figure(
    data_path=DATA_PATH,
    output_path=OUTPUT_PATH,
    exclude_warmup=True,
):
    """
    Create the full 3 x 4 licking summary figure.

    Returns
    -------
    df : pandas.DataFrame
        Prepared trial dataframe.
    fig : matplotlib.figure.Figure
        Figure object.
    stats_df : pandas.DataFrame
        Full statistical result table.
    """
    df = prepare_licking_dataframe(data_path, exclude_warmup=exclude_warmup)
    results = collect_statistics(df)
    rng = np.random.default_rng(RNG_SEED)

    df_c1 = df[df["block_type"].ne(0)].copy()
    df_c3 = df_c1[df_c1["outcome_strict"].isin(OUTCOME_LEVELS)].copy()
    df_c4 = df[df["col4_x"].ne("Exclude") & df["outcome_strict"].isin(OUTCOME_LEVELS)].copy()

    order_c2 = ["SB\nCom", "SB\nRare", "LB\nCom", "LB\nRare", "NB\nShort", "NB\nLong"]
    order_c4 = ["SB Common", "SB Rare", "LB Common", "LB Rare"]

    fig, axes = plt.subplots(3, 4, figsize=(24, 18))

    plot_session_dots(
        axes[0, 0], df_c1, "trial_name", LATENCY_COL, ["Short", "Long"],
        title="Col 1: Latency\n(Excl. Neutral Block)", ylabel="Latency (s)", rng=rng,
    )
    plot_session_dots(
        axes[1, 0], df_c1, "trial_name", COUNT_COL, ["Short", "Long"],
        title="Col 1: Lick Counts\n(Excl. Neutral Block)", ylabel="Count", rng=rng,
    )
    add_stats_box(axes[2, 0], stats_box_text(results, "col1"), fontsize=10)

    plot_session_dots(
        axes[0, 1], df, "col2_x", LATENCY_COL, order_c2,
        title="Col 2: Latency by Block/Rarity", ylabel="Latency (s)",
        separators=[1.5, 3.5], rng=rng,
    )
    plot_session_dots(
        axes[1, 1], df, "col2_x", COUNT_COL, order_c2,
        title="Col 2: Counts by Block/Rarity", ylabel="Count",
        separators=[1.5, 3.5], rng=rng,
    )
    add_stats_box(axes[2, 1], stats_box_text(results, "col2"), fontsize=9.5)

    plot_session_dots(
        axes[0, 2], df_c3, "trial_name", LATENCY_COL, ["Short", "Long"],
        hue_col="outcome_strict", hue_order=OUTCOME_LEVELS,
        title="Col 3: Latency by Outcome", ylabel="Latency (s)", rng=rng,
    )
    plot_session_dots(
        axes[1, 2], df_c3, "trial_name", COUNT_COL, ["Short", "Long"],
        hue_col="outcome_strict", hue_order=OUTCOME_LEVELS,
        title="Col 3: Counts by Outcome", ylabel="Count", rng=rng,
    )
    add_stats_box(axes[2, 2], stats_box_text(results, "col3"), fontsize=9.5)

    plot_session_dots(
        axes[0, 3], df_c4, "col4_x", LATENCY_COL, order_c4,
        hue_col="outcome_strict", hue_order=OUTCOME_LEVELS,
        title="Col 4: Latency by Block/Outcome", ylabel="Latency (s)",
        rotation=45, rng=rng,
    )
    plot_session_dots(
        axes[1, 3], df_c4, "col4_x", COUNT_COL, order_c4,
        hue_col="outcome_strict", hue_order=OUTCOME_LEVELS,
        title="Col 4: Counts by Block/Outcome", ylabel="Count",
        rotation=45, rng=rng,
    )
    add_stats_box(axes[2, 3], stats_box_text(results, "col4"), fontsize=8.6)

    fig.tight_layout(h_pad=2.0, w_pad=1.6)

    if output_path:
        fig.savefig(output_path, dpi=600, bbox_inches="tight")

    stats_df = (
        pd.DataFrame(results)
        .sort_values(["value_col", "panel", "order"])
        .reset_index(drop=True)
    )

    return df, fig, stats_df


df, fig, stats_df = make_licking_analysis_figure(
    data_path=DATA_PATH,
    output_path=OUTPUT_PATH,
    exclude_warmup=True,
)

plt.show()
stats_df


,comparison,value_col,test,statistic,p_raw,p_holm,n_trials,n_sessions,mean_diff,effect_dz,normality_p,panel,section,order
0,Short vs Long,all_lick_count,N/A,NaN,NaN,NaN,239,1,NaN,NaN,NaN,col1,COUNTS,10
1,SB Com vs SB Rare,all_lick_count,N/A,NaN,NaN,NaN,120,1,NaN,NaN,NaN,col2,COUNTS,11
2,LB Com vs LB Rare,all_lick_count,N/A,NaN,NaN,NaN,119,1,NaN,NaN,NaN,col2,COUNTS,12
3,NB Short vs NB Long,all_lick_count,N/A,NaN,NaN,NaN,37,1,NaN,NaN,NaN,col2,COUNTS,13
4,Short: Rew vs Pun,all_lick_count,N/A,NaN,NaN,NaN,126,1,NaN,NaN,NaN,col3,COUNTS,14
5,Long: Rew vs Pun,all_lick_count,N/A,NaN,NaN,NaN,113,1,NaN,NaN,NaN,col3,COUNTS,15
6,SB C: Rew vs Pun,all_lick_count,N/A,NaN,NaN,NaN,101,1,NaN,NaN,NaN,col4,COUNTS,16
7,SB R: Rew vs Pun,all_lick_count,N/A,NaN,NaN,NaN,19,0,NaN,NaN,NaN,col4,COUNTS,17
8,LB C: Rew vs Pun,all_lick_count,N/A,NaN,NaN,NaN,94,1,NaN,NaN,NaN,col4,COUNTS,18
9,LB R: Rew vs Pun,all_lick_count,N/A,NaN,NaN,NaN,25,1,NaN,NaN,NaN,col4,COUNTS,19


In [3]:
import ast
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

DATA_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/all_subjects_final_data.csv"
OUTPUT_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/licking_latency_by_isi_mean_sem.pdf"
N_BINS_PER_TYPE = 4

SESSION_COL = "session_idx"
ISI_COL = "trial_isi"
TRIAL_TYPE_COL = "trial_type"
BLOCK_COL = "block_type"
OUTCOME_COL = "outcome"
LATENCY_COL = "lick_reaction_time"
LICK_SIDE_COL = "lick_side"
LICK_DETAILS_COL = "all_lick_details"
SIDE_GROUP_COL = "side_group"
SIDE_PANEL_DATA_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/all_subjects_final_data.csv"

COLORS = {
    "Neutral block": "0.45",
    "Short/Long blocks": "black",
    "Short block": "#BCAE3E",
    "Long block": "blue",
    "Common": "#2ca02c",
    "Rare": "#ff7f0e",
    "Reward": "#1f77b4",
    "Punish": "#d62728",
    "Left": "#0057b8",
    "Right": "#b30000",
}


def parse_list_cell(value):
    """Parse a JSON/list-like CSV cell into a Python list."""
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            try:
                parsed = ast.literal_eval(text)
            except (SyntaxError, ValueError, TypeError):
                return []
        return parsed if isinstance(parsed, list) else []
    return []


def first_post_choice_side_from_details(value):
    """Return the physical side of the first lick with reaction_time >= 0."""
    for lick in parse_list_cell(value):
        if not isinstance(lick, dict):
            continue
        try:
            reaction_time = float(lick.get("reaction_time", np.nan))
        except (TypeError, ValueError):
            continue
        if np.isfinite(reaction_time) and reaction_time >= 0:
            side = lick.get("side")
            if side in {"Left", "Right"}:
                return side
    return np.nan


def add_physical_lick_side(df):
    """Add physical_lick_side aligned to the scalar first post-choice latency."""
    if LICK_DETAILS_COL in df.columns:
        df["physical_lick_side"] = df[LICK_DETAILS_COL].apply(first_post_choice_side_from_details)
    elif LICK_SIDE_COL in df.columns:
        df["physical_lick_side"] = df[LICK_SIDE_COL]
    else:
        df["physical_lick_side"] = np.nan

    if LICK_SIDE_COL in df.columns:
        df["physical_lick_side"] = df["physical_lick_side"].fillna(df[LICK_SIDE_COL])

    df.loc[~df["physical_lick_side"].isin(["Left", "Right"]), "physical_lick_side"] = np.nan
    return df


def add_side_group(df):
    """Label the side-panel cohorts: pooled reverse mice and normal YH24LG."""
    if "subject" not in df.columns:
        df[SIDE_GROUP_COL] = "Dataset"
        return df

    subject = df["subject"].astype(str)
    df[SIDE_GROUP_COL] = pd.Series(np.nan, index=df.index, dtype="object")
    df.loc[subject.isin(["MC06", "MC07", "MC08"]), SIDE_GROUP_COL] = "MC06-08 Reverse"
    df.loc[subject.isin(["YH24LG", "MC16", "MC17"]), SIDE_GROUP_COL] = "YH24LG/MC16/MC17 Normal"
    return df


def prepare_side_latency_dataframe(path=None, source_df=None, exclude_warmup=True):
    """Prepare pooled data used only for physical left/right side panels."""
    if source_df is None:
        if path is None:
            path = SIDE_PANEL_DATA_PATH
        df = pd.read_csv(path).copy()
    else:
        df = source_df.copy()

    required = {ISI_COL, TRIAL_TYPE_COL, BLOCK_COL, LATENCY_COL}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns for side panels: {sorted(missing)}")

    if SESSION_COL not in df.columns:
        df[SESSION_COL] = 1

    if "session_uid" in df.columns:
        session_codes, _ = pd.factorize(df["session_uid"].astype(str), sort=False)
        df[SESSION_COL] = session_codes + 1

    for col in [SESSION_COL, ISI_COL, TRIAL_TYPE_COL, BLOCK_COL, LATENCY_COL]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if exclude_warmup and "warm_up" in df.columns:
        df = df[df["warm_up"].eq(0)].copy()

    df = add_physical_lick_side(df)
    df = add_side_group(df)

    df = df.dropna(
        subset=[SESSION_COL, ISI_COL, TRIAL_TYPE_COL, BLOCK_COL, LATENCY_COL, "physical_lick_side", SIDE_GROUP_COL]
    ).copy()
    df = df[df[SIDE_GROUP_COL].isin(["MC06-08 Reverse", "YH24LG/MC16/MC17 Normal"])].copy()

    df[TRIAL_TYPE_COL] = df[TRIAL_TYPE_COL].astype(int)
    df[BLOCK_COL] = df[BLOCK_COL].astype(int)

    return df


def prepare_latency_dataframe(path, exclude_warmup=True):
    """Load data and add trial/block/outcome labels for latency by ISI analysis."""
    df = pd.read_csv(path).copy()

    required = {ISI_COL, TRIAL_TYPE_COL, BLOCK_COL, OUTCOME_COL, LATENCY_COL}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    if SESSION_COL not in df.columns:
        df[SESSION_COL] = 1

    for col in [SESSION_COL, ISI_COL, TRIAL_TYPE_COL, BLOCK_COL, LATENCY_COL]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if exclude_warmup and "warm_up" in df.columns:
        df = df[df["warm_up"].eq(0)].copy()

    df = add_physical_lick_side(df)

    df = df.dropna(
        subset=[SESSION_COL, ISI_COL, TRIAL_TYPE_COL, BLOCK_COL, LATENCY_COL]
    ).copy()

    df[TRIAL_TYPE_COL] = df[TRIAL_TYPE_COL].astype(int)
    df[BLOCK_COL] = df[BLOCK_COL].astype(int)

    df["trial_name"] = df[TRIAL_TYPE_COL].map({1: "Short", 2: "Long"})
    df["block_name"] = df[BLOCK_COL].map({
        0: "Neutral",
        1: "Short Block",
        2: "Long Block",
    })

    df["outcome_strict"] = df[OUTCOME_COL].replace({
        "RewardNaive": "Reward",
        "PunishNaive": "Punish",
    })

    df["is_common"] = np.nan
    df.loc[(df[BLOCK_COL] == 1) & (df[TRIAL_TYPE_COL] == 1), "is_common"] = 1
    df.loc[(df[BLOCK_COL] == 1) & (df[TRIAL_TYPE_COL] == 2), "is_common"] = 0
    df.loc[(df[BLOCK_COL] == 2) & (df[TRIAL_TYPE_COL] == 2), "is_common"] = 1
    df.loc[(df[BLOCK_COL] == 2) & (df[TRIAL_TYPE_COL] == 1), "is_common"] = 0

    return df


def get_boundary(df, default=1.25):
    """Get ISI boundary from isi_devider if present."""
    if "isi_devider" not in df.columns:
        return default

    values = pd.to_numeric(df["isi_devider"], errors="coerce").dropna()
    return float(values.median()) if not values.empty else default


def assign_psychometric_isi_bins(data, n_bins_per_type=4):
    """
    Bin ISIs using the same logic as the psychometric code:
    separately rank/qcut short and long trial types.
    """
    parts = []
    n_bins_per_type = max(int(n_bins_per_type), 1)

    for trial_type in [1, 2]:
        tt = data[data[TRIAL_TYPE_COL].eq(trial_type)].dropna(
            subset=[ISI_COL, LATENCY_COL]
        ).copy()

        if tt.empty:
            continue

        n_bins = min(n_bins_per_type, len(tt))
        ranked_isi = tt[ISI_COL].rank(method="first")

        try:
            tt["isi_bin"] = pd.qcut(
                ranked_isi,
                q=n_bins,
                labels=False,
                duplicates="drop",
            )
        except ValueError:
            continue

        tt = tt.dropna(subset=["isi_bin"]).copy()
        tt["isi_bin"] = tt["isi_bin"].astype(int)
        parts.append(tt)

    if not parts:
        return pd.DataFrame(columns=list(data.columns) + ["isi_bin"])

    return pd.concat(parts, ignore_index=True)


def binned_latency_mean_sem(data, n_bins_per_type=4):
    """
    For each ISI bin, average trials within session first.
    Then plot mean +/- SEM across sessions.
    """
    binned = assign_psychometric_isi_bins(data, n_bins_per_type=n_bins_per_type)

    if binned.empty:
        return pd.DataFrame()

    group_cols = [TRIAL_TYPE_COL, "isi_bin"]

    raw_summary = (
        binned
        .groupby(group_cols, observed=True)
        .agg(
            isi_mean=(ISI_COL, "mean"),
            n_trials=(LATENCY_COL, "size"),
        )
        .reset_index()
    )

    session_bin = (
        binned
        .groupby([SESSION_COL] + group_cols, observed=True)
        .agg(session_latency_mean=(LATENCY_COL, "mean"))
        .reset_index()
    )

    session_summary = (
        session_bin
        .groupby(group_cols, observed=True)
        .agg(
            latency_mean=("session_latency_mean", "mean"),
            latency_sem=("session_latency_mean", "sem"),
            n_sessions=("session_latency_mean", "count"),
        )
        .reset_index()
    )

    summary = raw_summary.merge(session_summary, on=group_cols, how="inner")
    summary["latency_sem"] = summary["latency_sem"].fillna(0)

    return summary.sort_values("isi_mean").reset_index(drop=True)


def style_axis(ax):
    """Simple plot styling with grids off."""
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_axisbelow(True)
    ax.set_box_aspect(1)
    ax.set_xlabel("ISI (s)")
    ax.set_ylabel("Latency (s)")


def physical_side_reward_label(side_group, side):
    """Describe the task meaning of a physical lick side for each cohort."""
    if side_group == "MC06-08 Reverse":
        reward_label = "Long rewarded" if side == "Left" else "Short rewarded"
    elif side_group == "YH24LG/MC16/MC17 Normal":
        reward_label = "Short rewarded" if side == "Left" else "Long rewarded"
    else:
        reward_label = "reward mapping unknown"
    return f"{side_group} {side} ({reward_label})"


def physical_side_traces(data, side_group, side_suffix=""):
    """Build left/right traces for one side-panel cohort."""
    group_data = data[data[SIDE_GROUP_COL].eq(side_group)]
    return [
        {
            "data": group_data[group_data["physical_lick_side"].eq("Left")],
            "label": physical_side_reward_label(side_group, "Left") + side_suffix,
            "color": COLORS["Left"],
            "linestyle": "-" if side_group == "MC06-08 Reverse" else "--",
        },
        {
            "data": group_data[group_data["physical_lick_side"].eq("Right")],
            "label": physical_side_reward_label(side_group, "Right") + side_suffix,
            "color": COLORS["Right"],
            "linestyle": "-" if side_group == "MC06-08 Reverse" else "--",
        },
    ]


def plot_latency_trace(ax, data, label, color, n_bins_per_type=4, linestyle="-"):
    """Plot one mean +/- SEM trace without showing individual data dots."""
    summary = binned_latency_mean_sem(data, n_bins_per_type=n_bins_per_type)

    if summary.empty:
        return None

    ax.errorbar(
        summary["isi_mean"],
        summary["latency_mean"],
        yerr=summary["latency_sem"],
        fmt=linestyle,
        color=color,
        linewidth=2,
        elinewidth=1.3,
        capsize=4,
        capthick=1.3,
        label=label,
    )

    return summary.assign(trace=label)


def plot_latency_panel(ax, traces, title, boundary=None, n_bins_per_type=4):
    """Plot all traces for one panel."""
    summaries = []

    for trace in traces:
        trace_data = trace["data"]
        session_column = "session_uid" if "session_uid" in trace_data.columns else SESSION_COL
        n_subjects = trace_data["subject"].nunique() if "subject" in trace_data.columns else 0
        n_sessions = trace_data[session_column].nunique() if session_column in trace_data.columns else 0
        count_label = (
            f'{trace["label"]}\n'
            f'(n={len(trace_data):,} trials; {n_subjects} subjects; {n_sessions} sessions)'
        )
        summary = plot_latency_trace(
            ax=ax,
            data=trace_data,
            label=count_label,
            color=trace["color"],
            n_bins_per_type=n_bins_per_type,
            linestyle=trace.get("linestyle", "-"),
        )
        if summary is not None:
            summaries.append(summary)

    if boundary is not None:
        ax.axvline(boundary, color="k", linestyle="--", alpha=0.25, linewidth=1)

    ax.set_title(title, fontweight="bold")
    style_axis(ax)

    handles, _ = ax.get_legend_handles_labels()
    if handles:
        ax.legend(
            frameon=False,
            fontsize=6.2,
            loc="best",
            borderaxespad=0.3,
        )
    else:
        ax.text(0.5, 0.5, "No data", transform=ax.transAxes, ha="center", va="center")

    if summaries:
        return pd.concat(summaries, ignore_index=True)

    return pd.DataFrame()


def make_latency_by_isi_figure(
    data_path=DATA_PATH,
    output_path=OUTPUT_PATH,
    n_bins_per_type=N_BINS_PER_TYPE,
    exclude_warmup=True,
    side_data_path=None,
    side_df=None,
):
    """Create the latency-by-ISI figure, optionally adding physical side panels."""
    df = prepare_latency_dataframe(data_path, exclude_warmup=exclude_warmup)
    boundary = get_boundary(df)

    side_data = None
    if side_df is not None or side_data_path is not None:
        side_data = prepare_side_latency_dataframe(
            path=side_data_path,
            source_df=side_df,
            exclude_warmup=exclude_warmup,
        )
        side_boundary = get_boundary(side_data, default=boundary)
    else:
        side_boundary = boundary

    n_side_panels = 3 if side_data is not None else 0
    ncols = 7 + n_side_panels

    fig = plt.figure(figsize=(5.3 * ncols, 6.5), constrained_layout=True)
    gs = gridspec.GridSpec(
        nrows=1,
        ncols=ncols,
        figure=fig,
        width_ratios=[1] * ncols,
        wspace=0.35,
    )

    ax_all = fig.add_subplot(gs[0, 0])
    ax_common = fig.add_subplot(gs[0, 1])
    ax_common_outcome = fig.add_subplot(gs[0, 2])
    ax_rare_outcome = fig.add_subplot(gs[0, 3])
    ax_outcome = fig.add_subplot(gs[0, 4])
    ax_short_block = fig.add_subplot(gs[0, 5])
    ax_long_block = fig.add_subplot(gs[0, 6])

    df_non_neutral = df[df[BLOCK_COL].isin([1, 2])].copy()
    df_outcome = df_non_neutral[
        df_non_neutral["outcome_strict"].isin(["Reward", "Punish"])
    ].copy()

    summaries = {}

    summaries["all_trials"] = plot_latency_panel(
        ax_all,
        traces=[
            {
                "data": df[df[BLOCK_COL].eq(0)],
                "label": "Neutral block",
                "color": COLORS["Neutral block"],
            },
            {
                "data": df[df[BLOCK_COL].isin([1, 2])],
                "label": "Short/Long blocks",
                "color": COLORS["Short/Long blocks"],
            },
        ],
        title="All Trials",
        boundary=boundary,
        n_bins_per_type=n_bins_per_type,
    )

    summaries["common_vs_rare"] = plot_latency_panel(
        ax_common,
        traces=[
            {
                "data": df_non_neutral[df_non_neutral["is_common"].eq(1)],
                "label": "Common",
                "color": COLORS["Common"],
            },
            {
                "data": df_non_neutral[df_non_neutral["is_common"].eq(0)],
                "label": "Rare",
                "color": COLORS["Rare"],
            },
        ],
        title="Common vs Rare",
        boundary=boundary,
        n_bins_per_type=n_bins_per_type,
    )

    df_common_outcome = df_outcome[df_outcome["is_common"].eq(1)].copy()
    summaries["common_reward_vs_punish"] = plot_latency_panel(
        ax_common_outcome,
        traces=[
            {
                "data": df_common_outcome[df_common_outcome["outcome_strict"].eq("Reward")],
                "label": "Reward",
                "color": COLORS["Reward"],
            },
            {
                "data": df_common_outcome[df_common_outcome["outcome_strict"].eq("Punish")],
                "label": "Punish",
                "color": COLORS["Punish"],
            },
        ],
        title="Common Trials\nReward vs Punish",
        boundary=boundary,
        n_bins_per_type=n_bins_per_type,
    )

    df_rare_outcome = df_outcome[df_outcome["is_common"].eq(0)].copy()
    summaries["rare_reward_vs_punish"] = plot_latency_panel(
        ax_rare_outcome,
        traces=[
            {
                "data": df_rare_outcome[df_rare_outcome["outcome_strict"].eq("Reward")],
                "label": "Reward",
                "color": COLORS["Reward"],
            },
            {
                "data": df_rare_outcome[df_rare_outcome["outcome_strict"].eq("Punish")],
                "label": "Punish",
                "color": COLORS["Punish"],
            },
        ],
        title="Rare Trials\nReward vs Punish",
        boundary=boundary,
        n_bins_per_type=n_bins_per_type,
    )

    summaries["reward_vs_punish_non_neutral"] = plot_latency_panel(
        ax_outcome,
        traces=[
            {
                "data": df_outcome[df_outcome["outcome_strict"].eq("Reward")],
                "label": "Reward",
                "color": COLORS["Reward"],
            },
            {
                "data": df_outcome[df_outcome["outcome_strict"].eq("Punish")],
                "label": "Punish",
                "color": COLORS["Punish"],
            },
        ],
        title="Reward vs Punish\nExcl. Neutral",
        boundary=boundary,
        n_bins_per_type=n_bins_per_type,
    )

    df_short_block = df_outcome[df_outcome[BLOCK_COL].eq(1)].copy()
    summaries["short_block_reward_vs_punish"] = plot_latency_panel(
        ax_short_block,
        traces=[
            {
                "data": df_short_block[df_short_block["outcome_strict"].eq("Reward")],
                "label": "Reward",
                "color": COLORS["Reward"],
            },
            {
                "data": df_short_block[df_short_block["outcome_strict"].eq("Punish")],
                "label": "Punish",
                "color": COLORS["Punish"],
            },
        ],
        title="Short Block\nReward vs Punish",
        boundary=boundary,
        n_bins_per_type=n_bins_per_type,
    )

    df_long_block = df_outcome[df_outcome[BLOCK_COL].eq(2)].copy()
    summaries["long_block_reward_vs_punish"] = plot_latency_panel(
        ax_long_block,
        traces=[
            {
                "data": df_long_block[df_long_block["outcome_strict"].eq("Reward")],
                "label": "Reward",
                "color": COLORS["Reward"],
            },
            {
                "data": df_long_block[df_long_block["outcome_strict"].eq("Punish")],
                "label": "Punish",
                "color": COLORS["Punish"],
            },
        ],
        title="Long Block\nReward vs Punish",
        boundary=boundary,
        n_bins_per_type=n_bins_per_type,
    )

    if side_data is not None:
        side_panels = [
            ("physical_side_all_blocks", side_data, "Physical Side\nAll Blocks"),
            ("physical_side_short_block", side_data[side_data[BLOCK_COL].eq(1)], "Physical Side\nShort Block"),
            ("physical_side_long_block", side_data[side_data[BLOCK_COL].eq(2)], "Physical Side\nLong Block"),
        ]

        for offset, (summary_key, panel_data, title) in enumerate(side_panels, start=7):
            ax_side = fig.add_subplot(gs[0, offset])
            traces = []
            for side_group in ["MC06-08 Reverse", "YH24LG/MC16/MC17 Normal"]:
                traces.extend(physical_side_traces(panel_data, side_group))
            summaries[summary_key] = plot_latency_panel(
                ax_side,
                traces=traces,
                title=title,
                boundary=side_boundary,
                n_bins_per_type=n_bins_per_type,
            )

    fig.suptitle("Lick Latency by ISI Bin: Mean +/- SEM", fontsize=16, fontweight="bold")

    if output_path:
        fig.savefig(output_path, dpi=600, bbox_inches="tight")

    return df, fig, summaries


df_latency, fig, latency_summaries = make_latency_by_isi_figure()
plt.show()

In [4]:
import ast
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D

DATA_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/all_subjects_final_data.csv"
OUTPUT_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/licking_raster_and_average_trajectory.pdf"

SESSION_COL = "session_idx"
LICK_TIMES_COL = "all_lick_reaction_times"  # already aligned to spout movement
LICK_DETAILS_COL = "all_lick_details"
FIRST_LICK_COL = "lick_reaction_time"
LICK_SIDE_COL = "lick_side"

RASTER_WINDOW = (-1.0, 3.0)
BIN_SIZE = 0.025
EXCLUDE_WARMUP = True

COLORS = {
    "All": "black",
    "Common": "#2ca02c",
    "Rare": "#ff7f0e",
    "Reward": "#1f77b4",
    "Punish": "#d62728",
    "Long trial": "#9467bd",
    "Short trial": "#17becf",
}

SIDE_COLORS = {
    "Left": "#74a9cf",
    "Right": "#f28e8c",
}

FIRST_LICK_COLORS = {
    "Left": "#0057b8",
    "Right": "#b30000",
}

UNKNOWN_LICK_COLOR = "0.60"
SEPARATOR_COLOR = "0.15"


def parse_list_cell(value):
    """Parse JSON/Python-list CSV cells into a Python list."""
    if isinstance(value, list):
        return value

    if value is None:
        return []

    if isinstance(value, float) and pd.isna(value):
        return []

    if isinstance(value, np.ndarray):
        return value.tolist()

    if not isinstance(value, str):
        return []

    text = value.strip()
    if not text:
        return []

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        try:
            parsed = ast.literal_eval(text)
        except (ValueError, SyntaxError, TypeError):
            return []

    return parsed if isinstance(parsed, list) else []


def to_float_or_nan(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def normalize_lick_details(row):
    """Return one trial's lick events with side and aligned reaction time."""
    details = parse_list_cell(row.get(LICK_DETAILS_COL, []))
    normalized = []

    for lick in details:
        if not isinstance(lick, dict):
            continue

        rt = to_float_or_nan(
            lick.get("reaction_time", lick.get("start_time_from_choice"))
        )
        if not np.isfinite(rt):
            continue

        side = lick.get("side")
        if side not in SIDE_COLORS:
            side = None

        normalized.append({
            "reaction_time": rt,
            "side": side,
            "lick_index": lick.get("lick_index"),
        })

    if normalized:
        normalized.sort(key=lambda lick: lick["reaction_time"])
        return normalized

    # Fallback for older CSVs that have only all_lick_reaction_times.
    fallback_times = parse_list_cell(row.get(LICK_TIMES_COL, []))
    fallback_side = row.get(LICK_SIDE_COL) if row.get(LICK_SIDE_COL) in SIDE_COLORS else None
    for lick_index, value in enumerate(fallback_times, start=1):
        rt = to_float_or_nan(value)
        if np.isfinite(rt):
            normalized.append({
                "reaction_time": rt,
                "side": fallback_side if lick_index == 1 else None,
                "lick_index": lick_index,
            })

    if not normalized and pd.notna(row.get(FIRST_LICK_COL, np.nan)):
        normalized.append({
            "reaction_time": float(row[FIRST_LICK_COL]),
            "side": fallback_side,
            "lick_index": 1,
        })

    normalized.sort(key=lambda lick: lick["reaction_time"])
    return normalized


def first_post_choice_lick(licks):
    """Use the first lick at or after spout movement as latency/order anchor."""
    for lick in licks:
        if lick["reaction_time"] >= 0:
            return lick
    return None


def prepare_lick_dataframe(path):
    """Load data and derive lick-event arrays for side-coded rasters."""
    df = pd.read_csv(path).copy()

    required = {"trial_isi", "trial_type", "block_type", "outcome"}
    if LICK_DETAILS_COL not in df.columns and LICK_TIMES_COL not in df.columns:
        required.add(LICK_DETAILS_COL)

    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    if SESSION_COL not in df.columns:
        df[SESSION_COL] = 1

    if FIRST_LICK_COL not in df.columns:
        df[FIRST_LICK_COL] = np.nan

    if LICK_SIDE_COL not in df.columns:
        df[LICK_SIDE_COL] = None

    if LICK_DETAILS_COL not in df.columns:
        df[LICK_DETAILS_COL] = "[]"

    if LICK_TIMES_COL not in df.columns:
        df[LICK_TIMES_COL] = "[]"

    if EXCLUDE_WARMUP and "warm_up" in df.columns:
        df = df[df["warm_up"].eq(0)].copy()

    for col in ["trial_isi", "trial_type", "block_type", FIRST_LICK_COL, SESSION_COL]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["trial_isi", "trial_type", "block_type"]).copy()
    df["trial_type"] = df["trial_type"].astype(int)
    df["block_type"] = df["block_type"].astype(int)

    df["trial_name"] = df["trial_type"].map({1: "Short", 2: "Long"})
    df["block_name"] = df["block_type"].map({
        0: "Neutral",
        1: "Short Block",
        2: "Long Block",
    })

    df["outcome_strict"] = df["outcome"].replace({
        "RewardNaive": "Reward",
        "PunishNaive": "Punish",
    })

    df["is_common"] = np.nan
    df.loc[(df["block_type"] == 1) & (df["trial_type"] == 1), "is_common"] = 1
    df.loc[(df["block_type"] == 1) & (df["trial_type"] == 2), "is_common"] = 0
    df.loc[(df["block_type"] == 2) & (df["trial_type"] == 2), "is_common"] = 1
    df.loc[(df["block_type"] == 2) & (df["trial_type"] == 1), "is_common"] = 0
    df["rarity"] = df["is_common"].map({1.0: "Common", 0.0: "Rare"})

    lick_details = []
    lick_times = []
    first_lick_latencies = []
    first_lick_sides = []

    for _, row in df.iterrows():
        licks = normalize_lick_details(row)
        licks_in_window = [
            lick for lick in licks
            if RASTER_WINDOW[0] <= lick["reaction_time"] <= RASTER_WINDOW[1]
        ]
        first_lick = first_post_choice_lick(licks_in_window)

        lick_details.append(licks_in_window)
        lick_times.append(np.array([lick["reaction_time"] for lick in licks_in_window], dtype=float))
        first_lick_latencies.append(
            first_lick["reaction_time"] if first_lick is not None else np.nan
        )
        first_lick_sides.append(
            first_lick["side"] if first_lick is not None else None
        )

    df["lick_details_aligned"] = lick_details
    df["lick_times_aligned"] = lick_times
    df["first_lick_latency_from_all"] = first_lick_latencies
    df["first_lick_side_from_all"] = first_lick_sides
    df["first_lick_latency_order"] = pd.to_numeric(
        df["first_lick_latency_from_all"], errors="coerce"
    )

    return df


def sort_by_latency(data):
    """Sort trials by first post-choice lick latency, placing no-lick trials last."""
    data = data.copy()
    data["_latency_sort"] = data["first_lick_latency_order"].fillna(np.inf)
    return (
        data
        .sort_values(["_latency_sort", "trial_isi"], kind="mergesort")
        .drop(columns="_latency_sort")
        .reset_index(drop=True)
    )


def make_event_matrix(data, bin_edges, sort_by="trial_isi"):
    """
    Make raster matrix for group-colored panels.

    Empty no-lick bins are NaN for display. Lick bins are 1.
    """
    data = data.sort_values(sort_by).reset_index(drop=True)
    n_trials = len(data)
    n_bins = len(bin_edges) - 1

    matrix = np.full((n_trials, n_bins), np.nan)

    for i, times in enumerate(data["lick_times_aligned"]):
        counts, _ = np.histogram(times, bins=bin_edges)
        matrix[i, counts > 0] = 1.0

    return matrix, data


def make_rate_matrix(data, bin_edges):
    """
    Make trial x time-bin lick-rate matrix.

    For Hz, no-lick bins are 0, not NaN. Otherwise the average would ignore
    silent trials and overestimate lick rate.
    """
    n_trials = len(data)
    n_bins = len(bin_edges) - 1
    matrix = np.zeros((n_trials, n_bins), dtype=float)

    for i, times in enumerate(data["lick_times_aligned"]):
        counts, _ = np.histogram(times, bins=bin_edges)
        matrix[i, :] = counts / BIN_SIZE

    return matrix


def colored_nan_cmap(color):
    """Single-color colormap with NaNs transparent."""
    cmap = ListedColormap([color])
    cmap.set_bad((1, 1, 1, 0))
    return cmap


def style_axis(ax):
    """Clean plot style with grids off."""
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.axvline(0, color="k", linestyle="--", linewidth=1, alpha=0.45)


def add_side_legend(ax, show_first=True):
    handles = [
        Line2D([0], [0], marker="|", color=SIDE_COLORS["Left"], linestyle="None",
               markersize=7, markeredgewidth=1.3, label="Left lick"),
        Line2D([0], [0], marker="|", color=SIDE_COLORS["Right"], linestyle="None",
               markersize=7, markeredgewidth=1.3, label="Right lick"),
    ]
    if show_first:
        handles.extend([
            Line2D([0], [0], marker="|", color=FIRST_LICK_COLORS["Left"], linestyle="None",
                   markersize=10, markeredgewidth=2.1, label="First Left"),
            Line2D([0], [0], marker="|", color=FIRST_LICK_COLORS["Right"], linestyle="None",
                   markersize=10, markeredgewidth=2.1, label="First Right"),
        ])
    ax.legend(
        handles=handles,
        frameon=False,
        fontsize=6.5,
        loc="upper right",
        handlelength=0.8,
        handletextpad=0.3,
        borderaxespad=0.2,
    )


def sort_for_raster(data, sort_mode):
    """Sort trials within a raster group."""
    if sort_mode == "latency":
        return sort_by_latency(data)

    data = data.copy()
    data["_latency_sort"] = data["first_lick_latency_order"].fillna(np.inf)
    return (
        data
        .sort_values(["trial_isi", "_latency_sort"], kind="mergesort")
        .drop(columns="_latency_sort")
        .reset_index(drop=True)
    )


def plot_side_coded_raster_groups(ax, groups, title, sort_mode="isi", highlight_first=True):
    """Plot rasters where every lick event is color-coded by lick side."""
    y_offset = 0
    yticks = []
    ylabels = []

    # Plot bottom-to-top in reverse so the visible top-to-bottom order
    # matches the order requested in each panel definition.
    raster_groups = list(reversed(groups))

    for group_index, group in enumerate(raster_groups):
        sorted_data = sort_for_raster(group["data"], sort_mode=sort_mode)
        n_trials = len(sorted_data)

        if n_trials == 0:
            continue

        licks_by_side = {
            "Left": {"x": [], "y": []},
            "Right": {"x": [], "y": []},
        }
        first_by_side = {
            "Left": {"x": [], "y": []},
            "Right": {"x": [], "y": []},
        }
        unknown_x = []
        unknown_y = []
        unknown_first_x = []
        unknown_first_y = []

        for row_index, row in sorted_data.iterrows():
            y = y_offset + row_index + 0.5
            first_rt = row["first_lick_latency_from_all"]
            first_side = row["first_lick_side_from_all"]
            first_marked = False

            for lick in row["lick_details_aligned"]:
                rt = lick["reaction_time"]
                lick_side = lick.get("side")
                is_first = (
                    highlight_first
                    and np.isfinite(first_rt)
                    and np.isclose(rt, first_rt, atol=1e-9)
                    and not first_marked
                )

                if is_first:
                    side_key = first_side if first_side in FIRST_LICK_COLORS else lick_side
                    if side_key in FIRST_LICK_COLORS:
                        first_by_side[side_key]["x"].append(rt)
                        first_by_side[side_key]["y"].append(y)
                    else:
                        unknown_first_x.append(rt)
                        unknown_first_y.append(y)
                    first_marked = True
                    continue

                if lick_side in SIDE_COLORS:
                    licks_by_side[lick_side]["x"].append(rt)
                    licks_by_side[lick_side]["y"].append(y)
                else:
                    unknown_x.append(rt)
                    unknown_y.append(y)

        for side, values in licks_by_side.items():
            if values["x"]:
                ax.scatter(
                    values["x"],
                    values["y"],
                    marker="|",
                    s=7,
                    color=SIDE_COLORS[side],
                    alpha=0.45,
                    linewidths=0.7,
                )

        if unknown_x:
            ax.scatter(
                unknown_x,
                unknown_y,
                marker="|",
                s=7,
                color=UNKNOWN_LICK_COLOR,
                alpha=0.35,
                linewidths=0.65,
            )

        for side, values in first_by_side.items():
            if values["x"]:
                ax.scatter(
                    values["x"],
                    values["y"],
                    marker="|",
                    s=18,
                    color=FIRST_LICK_COLORS[side],
                    alpha=0.98,
                    linewidths=1.5,
                )

        if unknown_first_x:
            ax.scatter(
                unknown_first_x,
                unknown_first_y,
                marker="|",
                s=16,
                color="0.15",
                alpha=0.95,
                linewidths=1.25,
            )

        yticks.append(y_offset + n_trials / 2)
        ylabels.append(group["label"])
        y_offset += n_trials

        if group_index < len(raster_groups) - 1:
            ax.axhline(y_offset, color=SEPARATOR_COLOR, linewidth=1.1)

    ax.set_xlim(RASTER_WINDOW)
    ax.set_ylim(0, max(y_offset, 1))
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Time from spout movement (s)")
    ylabel = "Trials sorted by first lick latency" if sort_mode == "latency" else "Trials sorted by ISI"
    ax.set_ylabel(ylabel)
    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels)
    style_axis(ax)
    add_side_legend(ax, show_first=highlight_first)


def plot_latency_side_raster_groups(ax, groups, title):
    """Plot latency-sorted side-coded raster maps."""
    plot_side_coded_raster_groups(
        ax=ax,
        groups=groups,
        title=title,
        sort_mode="latency",
        highlight_first=True,
    )


def plot_raster_groups(ax, groups, title, bin_edges):
    """Plot ISI-sorted side-coded raster maps for one panel."""
    plot_side_coded_raster_groups(
        ax=ax,
        groups=groups,
        title=title,
        sort_mode="isi",
        highlight_first=False,
    )


def session_average_trajectory(data, bin_edges):
    """
    Average lick-rate trajectories.

    Step 1: count licks per bin for each trial and convert to Hz.
    Step 2: average trials of the same condition within each session.
    Step 3: average session means and compute SEM across sessions.
    """
    centers = bin_edges[:-1] + np.diff(bin_edges) / 2

    if data.empty:
        return centers, np.full_like(centers, np.nan), np.full_like(centers, np.nan)

    session_traces = []

    for _, session_data in data.groupby(SESSION_COL):
        rate_matrix = make_rate_matrix(session_data, bin_edges)

        if rate_matrix.size == 0:
            continue

        session_trace = np.nanmean(rate_matrix, axis=0)
        session_traces.append(session_trace)

    if len(session_traces) == 0:
        return centers, np.full_like(centers, np.nan), np.full_like(centers, np.nan)

    session_traces = np.vstack(session_traces)
    mean_trace = np.nanmean(session_traces, axis=0)

    if session_traces.shape[0] > 1:
        sem_trace = np.nanstd(session_traces, axis=0, ddof=1) / np.sqrt(session_traces.shape[0])
    else:
        sem_trace = np.zeros_like(mean_trace)

    return centers, mean_trace, sem_trace


def plot_average_groups(ax, groups, title, bin_edges):
    """Plot mean +/- SEM lick-rate trajectories with grids off."""
    for group in groups:
        x, mean_trace, sem_trace = session_average_trajectory(group["data"], bin_edges)

        ax.plot(
            x,
            mean_trace,
            color=group["color"],
            linewidth=2,
            label=group["label"],
        )

        ax.fill_between(
            x,
            mean_trace - sem_trace,
            mean_trace + sem_trace,
            color=group["color"],
            alpha=0.18,
            linewidth=0,
        )

    ax.set_xlim(RASTER_WINDOW)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Time from spout movement (s)")
    ax.set_ylabel("Lick rate (Hz)")
    ax.grid(False)
    ax.legend(frameon=False, fontsize=9)
    style_axis(ax)


def make_panel_groups(df):
    """Define raster/trajectory panels."""
    df_non_neutral = df[df["block_type"].isin([1, 2])].copy()
    df_outcome = df_non_neutral[
        df_non_neutral["outcome_strict"].isin(["Reward", "Punish"])
    ].copy()

    df_short_block = df_outcome[df_outcome["block_type"].eq(1)].copy()
    df_long_block = df_outcome[df_outcome["block_type"].eq(2)].copy()

    all_long = df[df["trial_type"].eq(2)].copy()
    all_short = df[df["trial_type"].eq(1)].copy()

    short_rare = df_short_block[df_short_block["is_common"].eq(0)].copy()
    short_common = df_short_block[df_short_block["is_common"].eq(1)].copy()
    long_rare = df_long_block[df_long_block["is_common"].eq(0)].copy()
    long_common = df_long_block[df_long_block["is_common"].eq(1)].copy()

    return [
        {
            "title": "All Trials\nLong vs Short",
            "raster_mode": "side_latency",
            "groups": [
                {
                    "label": "Long",
                    "data": all_long,
                    "color": COLORS["Long trial"],
                },
                {
                    "label": "Short",
                    "data": all_short,
                    "color": COLORS["Short trial"],
                },
            ],
        },
        {
            "title": "Common vs Rare",
            "raster_mode": "group",
            "groups": [
                {
                    "label": "Common",
                    "data": df_non_neutral[df_non_neutral["is_common"].eq(1)],
                    "color": COLORS["Common"],
                },
                {
                    "label": "Rare",
                    "data": df_non_neutral[df_non_neutral["is_common"].eq(0)],
                    "color": COLORS["Rare"],
                },
            ],
        },
        {
            "title": "Reward vs Punish\nExcl. Neutral",
            "raster_mode": "group",
            "groups": [
                {
                    "label": "Reward",
                    "data": df_outcome[df_outcome["outcome_strict"].eq("Reward")],
                    "color": COLORS["Reward"],
                },
                {
                    "label": "Punish",
                    "data": df_outcome[df_outcome["outcome_strict"].eq("Punish")],
                    "color": COLORS["Punish"],
                },
            ],
        },
        {
            "title": "Short Block\nReward vs Punish",
            "raster_mode": "group",
            "groups": [
                {
                    "label": "Reward",
                    "data": df_short_block[df_short_block["outcome_strict"].eq("Reward")],
                    "color": COLORS["Reward"],
                },
                {
                    "label": "Punish",
                    "data": df_short_block[df_short_block["outcome_strict"].eq("Punish")],
                    "color": COLORS["Punish"],
                },
            ],
        },
        {
            "title": "Long Block\nReward vs Punish",
            "raster_mode": "group",
            "groups": [
                {
                    "label": "Reward",
                    "data": df_long_block[df_long_block["outcome_strict"].eq("Reward")],
                    "color": COLORS["Reward"],
                },
                {
                    "label": "Punish",
                    "data": df_long_block[df_long_block["outcome_strict"].eq("Punish")],
                    "color": COLORS["Punish"],
                },
            ],
        },
        {
            "title": "Short Block\nRare vs Common",
            "raster_mode": "side_latency",
            "groups": [
                {
                    "label": "Rare",
                    "data": short_rare,
                    "color": COLORS["Rare"],
                },
                {
                    "label": "Common",
                    "data": short_common,
                    "color": COLORS["Common"],
                },
            ],
        },
        {
            "title": "Long Block\nRare vs Common",
            "raster_mode": "side_latency",
            "groups": [
                {
                    "label": "Rare",
                    "data": long_rare,
                    "color": COLORS["Rare"],
                },
                {
                    "label": "Common",
                    "data": long_common,
                    "color": COLORS["Common"],
                },
            ],
        },
    ]


def make_licking_raster_figure():
    """Create raster maps and average lick-rate trajectories."""
    df = prepare_lick_dataframe(DATA_PATH)
    panels = make_panel_groups(df)

    bin_edges = np.arange(
        RASTER_WINDOW[0],
        RASTER_WINDOW[1] + BIN_SIZE,
        BIN_SIZE,
    )

    fig = plt.figure(figsize=(37, 10), constrained_layout=True)
    gs = gridspec.GridSpec(
        nrows=2,
        ncols=len(panels),
        figure=fig,
        height_ratios=[2.2, 1.0],
        wspace=0.26,
        hspace=0.30,
    )

    for col, panel in enumerate(panels):
        ax_raster = fig.add_subplot(gs[0, col])
        ax_avg = fig.add_subplot(gs[1, col])

        if panel.get("raster_mode") == "side_latency":
            plot_latency_side_raster_groups(
                ax=ax_raster,
                groups=panel["groups"],
                title=panel["title"],
            )
        else:
            plot_raster_groups(
                ax=ax_raster,
                groups=panel["groups"],
                title=panel["title"],
                bin_edges=bin_edges,
            )

        plot_average_groups(
            ax=ax_avg,
            groups=panel["groups"],
            title="Average trajectory",
            bin_edges=bin_edges,
        )

    fig.suptitle(
        "Mouse Licking Aligned to Spout Movement",
        fontsize=16,
        fontweight="bold",
    )

    fig.savefig(OUTPUT_PATH, dpi=600, bbox_inches="tight")
    return df, fig


df_raster, fig = make_licking_raster_figure()
plt.show()


In [5]:
# Repeat the three licking analyses after excluding trials with any negative lick time.
# This filters only in memory; it does not write or modify any CSV or figure files.
import ast
import json
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.figure import Figure

NEGATIVE_FILTER_DATA_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/all_subjects_final_data.csv"
SIDE_NEGATIVE_FILTER_DATA_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/all_subjects_final_data.csv"
NEGATIVE_LICK_TIMES_COL = "all_lick_reaction_times"
NO_NEGATIVE_LICKING_SUMMARY_OUTPUT_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/licking_analysis_no_negative_lick_trials.pdf"
NO_NEGATIVE_LATENCY_ISI_OUTPUT_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/licking_latency_by_isi_no_negative_lick_trials.pdf"
NO_NEGATIVE_RASTER_OUTPUT_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/licking_raster_and_average_trajectory_no_negative_lick_trials.pdf"


def parse_lick_time_values(value):
    """Parse one CSV cell from all_lick_reaction_times into numeric lick times."""
    if isinstance(value, np.ndarray):
        values = value.tolist()
    elif isinstance(value, (list, tuple)):
        values = list(value)
    elif value is None or (isinstance(value, float) and pd.isna(value)):
        values = []
    elif isinstance(value, str):
        text = value.strip()
        if not text:
            values = []
        else:
            try:
                values = json.loads(text)
            except json.JSONDecodeError:
                try:
                    values = ast.literal_eval(text)
                except (ValueError, SyntaxError, TypeError):
                    values = []
    else:
        values = []

    numeric_values = []
    for item in values if isinstance(values, list) else []:
        try:
            numeric_values.append(float(item))
        except (TypeError, ValueError):
            continue
    return numeric_values


def has_negative_lick_time(value):
    """True when a trial has at least one lick time before WindowChoice."""
    return any(lick_time < 0 for lick_time in parse_lick_time_values(value))


def load_negative_lick_filtered_dataframe(path=NEGATIVE_FILTER_DATA_PATH):
    """Load the CSV and drop trials with any negative value in all_lick_reaction_times."""
    source_df = pd.read_csv(path)
    if NEGATIVE_LICK_TIMES_COL not in source_df.columns:
        raise ValueError(f"Missing required column: {NEGATIVE_LICK_TIMES_COL}")

    negative_lick_mask = source_df[NEGATIVE_LICK_TIMES_COL].apply(has_negative_lick_time)
    filtered_df = source_df.loc[~negative_lick_mask].copy()

    summary = pd.DataFrame({
        "dataset": [
            "Original CSV",
            "Excluded: at least one negative lick time",
            "Retained for filtered analyses",
        ],
        "trials": [len(source_df), int(negative_lick_mask.sum()), len(filtered_df)],
    })

    excluded_by_trial_type = (
        source_df.loc[negative_lick_mask]
        .groupby("trial_type", dropna=False)
        .size()
        .rename(index={1: "Short", 2: "Long"})
        .rename("excluded_trials")
        .reset_index()
    )

    return filtered_df, negative_lick_mask, summary, excluded_by_trial_type


@contextmanager
def read_csv_from_filtered_dataframe(filtered_df, target_path=NEGATIVE_FILTER_DATA_PATH):
    """Temporarily make existing analysis functions read the in-memory filtered table."""
    original_read_csv = pd.read_csv
    target_name = Path(target_path).name

    def filtered_read_csv(filepath, *args, **kwargs):
        try:
            same_file = Path(str(filepath)).name == target_name
        except TypeError:
            same_file = False

        if same_file:
            return filtered_df.copy()
        return original_read_csv(filepath, *args, **kwargs)

    pd.read_csv = filtered_read_csv
    try:
        yield
    finally:
        pd.read_csv = original_read_csv


@contextmanager
def temporary_globals(**updates):
    """Temporarily set globals used by earlier notebook plotting functions."""
    missing = object()
    previous = {name: globals().get(name, missing) for name in updates}
    globals().update(updates)
    try:
        yield
    finally:
        for name, value in previous.items():
            if value is missing:
                globals().pop(name, None)
            else:
                globals()[name] = value


@contextmanager
def suppress_figure_writes():
    """Prevent reused plotting functions from saving PNGs during this filtered rerun."""
    original_savefig = Figure.savefig

    def no_savefig(self, *args, **kwargs):
        return None

    Figure.savefig = no_savefig
    try:
        yield
    finally:
        Figure.savefig = original_savefig


NEGATIVE_FILTER_LATENCY_COLORS = {
    "Neutral block": "0.45",
    "Short/Long blocks": "black",
    "Short block": "#BCAE3E",
    "Long block": "blue",
    "Common": "#2ca02c",
    "Rare": "#ff7f0e",
    "Reward": "#1f77b4",
    "Punish": "#d62728",
    "Left": "#0057b8",
    "Right": "#b30000",
}


def negative_filter_latency_style_axis(ax):
    """Style used by the latency-by-ISI figure before later cells redefine style_axis."""
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_axisbelow(True)
    ax.set_box_aspect(1)
    ax.set_xlabel("ISI (s)")
    ax.set_ylabel("Latency (s)")


OUTCOME_SIDE_COLORS = {
    ("Reward", "Left"): "#0057b8",
    ("Reward", "Right"): "#17becf",
    ("Punish", "Left"): "#b30000",
    ("Punish", "Right"): "#f28e2b",
}
OUTCOME_SIDE_LINESTYLES = {"Left": "-", "Right": "--"}
EXPECTED_REVERSE_SUBJECTS = {'MC06'}
EXPECTED_NORMAL_SUBJECTS = set()


def summarize_contingency_subjects(data):
    """Summarize subject-to-contingency mapping for the requested double-check."""
    required = {"subject", "Contingency_idx"}
    if not required.issubset(data.columns):
        missing = sorted(required.difference(data.columns))
        raise ValueError(f"Missing columns needed for contingency check: {missing}")

    session_col = "session_uid" if "session_uid" in data.columns else SESSION_COL
    grouping_cols = ["subject", "Contingency_idx"]
    if "Contingency" in data.columns:
        grouping_cols.append("Contingency")

    return (
        data
        .groupby(grouping_cols, dropna=False)[session_col]
        .nunique()
        .rename("sessions")
        .reset_index()
        .sort_values(["Contingency_idx", "subject"])
        .reset_index(drop=True)
    )


def verify_expected_contingency_subjects(data):
    """Raise if this dataset no longer matches the expected mouse/contingency split."""
    contingency_idx = pd.to_numeric(data["Contingency_idx"], errors="coerce")
    subjects = data["subject"].astype(str)

    reverse_subjects = set(subjects[contingency_idx.eq(2)].dropna().unique())
    normal_subjects = set(subjects[contingency_idx.eq(1)].dropna().unique())

    if reverse_subjects != EXPECTED_REVERSE_SUBJECTS:
        raise ValueError(
            "Reverse contingency subjects do not match expectation. "
            f"Expected {sorted(EXPECTED_REVERSE_SUBJECTS)}, found {sorted(reverse_subjects)}."
        )
    if normal_subjects != EXPECTED_NORMAL_SUBJECTS:
        raise ValueError(
            "Normal contingency subjects do not match expectation. "
            f"Expected {sorted(EXPECTED_NORMAL_SUBJECTS)}, found {sorted(normal_subjects)}."
        )


def contingency_mask(data, contingency_idx, fallback_subjects):
    """Filter by contingency index, falling back to known subjects if needed."""
    if "Contingency_idx" in data.columns:
        return pd.to_numeric(data["Contingency_idx"], errors="coerce").eq(contingency_idx)
    if "subject" in data.columns:
        return data["subject"].astype(str).isin(fallback_subjects)
    return pd.Series(False, index=data.index)


def outcome_side_traces(data):
    """Build Reward/Punish x physical Left/Right traces for one panel."""
    traces = []
    panel_data = data[
        data["outcome_strict"].isin(["Reward", "Punish"])
        & data["physical_lick_side"].isin(["Left", "Right"])
    ].copy()

    for outcome, side, label in [
        ("Reward", "Left", "Rewarded Left"),
        ("Reward", "Right", "Rewarded Right"),
        ("Punish", "Left", "Punished Left"),
        ("Punish", "Right", "Punished Right"),
    ]:
        traces.append({
            "data": panel_data[
                panel_data["outcome_strict"].eq(outcome)
                & panel_data["physical_lick_side"].eq(side)
            ],
            "label": label,
            "color": OUTCOME_SIDE_COLORS[(outcome, side)],
            "linestyle": OUTCOME_SIDE_LINESTYLES[side],
        })

    return traces


def available_side_groups(data):
    """Return side-panel groups in a stable order, restricted to groups in this row."""
    if data is None or data.empty or SIDE_GROUP_COL not in data.columns:
        return []
    present = set(data[SIDE_GROUP_COL].dropna())
    return [group for group in ["MC06-08 Reverse", "YH24LG/MC16/MC17 Normal"] if group in present]


def plot_latency_grid_panel(ax, traces, title, boundary, n_bins_per_type, row_idx, col_idx, row_label):
    """Plot one compact panel and apply row/column labelling for the 14 x 3 grid."""
    summary = plot_latency_panel(
        ax=ax,
        traces=traces,
        title=title,
        boundary=boundary,
        n_bins_per_type=n_bins_per_type,
    )

    if row_idx > 0:
        ax.set_title("")
    if col_idx == 0:
        ax.set_ylabel(f"{row_label}\nLatency (s)", fontweight="bold")
    else:
        ax.set_ylabel("")
    if row_idx < 2:
        ax.set_xlabel("")

    return summary


def add_latency_grid_row(fig, gs, row_idx, row_label, row_df, row_side_data, n_bins_per_type, summaries, prefix):
    """Add one row of the 13-column latency-by-ISI grid."""
    row_boundary = get_boundary(row_df)
    side_boundary = get_boundary(row_side_data, default=row_boundary) if row_side_data is not None else row_boundary

    df_non_neutral = row_df[row_df[BLOCK_COL].isin([1, 2])].copy()
    df_outcome = df_non_neutral[df_non_neutral["outcome_strict"].isin(["Reward", "Punish"])].copy()
    df_common_outcome = df_outcome[df_outcome["is_common"].eq(1)].copy()
    df_rare_outcome = df_outcome[df_outcome["is_common"].eq(0)].copy()
    df_short_block = df_outcome[df_outcome[BLOCK_COL].eq(1)].copy()
    df_long_block = df_outcome[df_outcome[BLOCK_COL].eq(2)].copy()
    df_neutral_outcome = row_df[
        row_df[BLOCK_COL].eq(0)
        & row_df["outcome_strict"].isin(["Reward", "Punish"])
    ].copy()
    df_all_outcome = row_df[row_df["outcome_strict"].isin(["Reward", "Punish"])].copy()

    panel_specs = [
        (
            "all_trials",
            "All Trials",
            row_boundary,
            [
                {"data": row_df[row_df[BLOCK_COL].eq(0)], "label": "Neutral block", "color": COLORS["Neutral block"]},
                {"data": row_df[row_df[BLOCK_COL].eq(1)], "label": "Short block", "color": COLORS["Short block"]},
                {"data": row_df[row_df[BLOCK_COL].eq(2)], "label": "Long block", "color": COLORS["Long block"]},
            ],
        ),
        (
            "common_vs_rare",
            "Common vs Rare",
            row_boundary,
            [
                {"data": df_non_neutral[df_non_neutral["is_common"].eq(1)], "label": "Common", "color": COLORS["Common"]},
                {"data": df_non_neutral[df_non_neutral["is_common"].eq(0)], "label": "Rare", "color": COLORS["Rare"]},
            ],
        ),
        (
            "common_reward_vs_punish",
            "Common Trials\nReward vs Punish",
            row_boundary,
            [
                {"data": df_common_outcome[df_common_outcome["outcome_strict"].eq("Reward")], "label": "Reward", "color": COLORS["Reward"]},
                {"data": df_common_outcome[df_common_outcome["outcome_strict"].eq("Punish")], "label": "Punish", "color": COLORS["Punish"]},
            ],
        ),
        (
            "rare_reward_vs_punish",
            "Rare Trials\nReward vs Punish",
            row_boundary,
            [
                {"data": df_rare_outcome[df_rare_outcome["outcome_strict"].eq("Reward")], "label": "Reward", "color": COLORS["Reward"]},
                {"data": df_rare_outcome[df_rare_outcome["outcome_strict"].eq("Punish")], "label": "Punish", "color": COLORS["Punish"]},
            ],
        ),
        (
            "reward_vs_punish_non_neutral",
            "Reward vs Punish\nExcl. Neutral",
            row_boundary,
            [
                {"data": df_outcome[df_outcome["outcome_strict"].eq("Reward")], "label": "Reward", "color": COLORS["Reward"]},
                {"data": df_outcome[df_outcome["outcome_strict"].eq("Punish")], "label": "Punish", "color": COLORS["Punish"]},
            ],
        ),
        (
            "neutral_reward_vs_punish",
            "Neutral Block\nReward vs Punish",
            row_boundary,
            [
                {"data": df_neutral_outcome[df_neutral_outcome["outcome_strict"].eq("Reward")], "label": "Reward", "color": COLORS["Reward"]},
                {"data": df_neutral_outcome[df_neutral_outcome["outcome_strict"].eq("Punish")], "label": "Punish", "color": COLORS["Punish"]},
            ],
        ),
        (
            "short_block_reward_vs_punish",
            "Short Block\nReward vs Punish",
            row_boundary,
            [
                {"data": df_short_block[df_short_block["outcome_strict"].eq("Reward")], "label": "Reward", "color": COLORS["Reward"]},
                {"data": df_short_block[df_short_block["outcome_strict"].eq("Punish")], "label": "Punish", "color": COLORS["Punish"]},
            ],
        ),
        (
            "long_block_reward_vs_punish",
            "Long Block\nReward vs Punish",
            row_boundary,
            [
                {"data": df_long_block[df_long_block["outcome_strict"].eq("Reward")], "label": "Reward", "color": COLORS["Reward"]},
                {"data": df_long_block[df_long_block["outcome_strict"].eq("Punish")], "label": "Punish", "color": COLORS["Punish"]},
            ],
        ),
    ]

    side_panels = [
        ("physical_side_all_blocks", row_side_data, "Physical Side\nAll Blocks"),
        ("physical_side_short_block", row_side_data[row_side_data[BLOCK_COL].eq(1)] if row_side_data is not None else None, "Physical Side\nShort Block"),
        ("physical_side_long_block", row_side_data[row_side_data[BLOCK_COL].eq(2)] if row_side_data is not None else None, "Physical Side\nLong Block"),
    ]

    for summary_key, panel_data, title in side_panels:
        traces = []
        if panel_data is not None:
            for side_group in available_side_groups(panel_data):
                traces.extend(physical_side_traces(panel_data, side_group))
        panel_specs.append((summary_key, title, side_boundary, traces))

    panel_specs.extend([
        (
            "outcome_side_all_blocks",
            "Reward/Punish x Side\nAll Blocks",
            row_boundary,
            outcome_side_traces(df_all_outcome),
        ),
        (
            "outcome_side_short_block",
            "Reward/Punish x Side\nShort Block",
            row_boundary,
            outcome_side_traces(df_all_outcome[df_all_outcome[BLOCK_COL].eq(1)]),
        ),
        (
            "outcome_side_long_block",
            "Reward/Punish x Side\nLong Block",
            row_boundary,
            outcome_side_traces(df_all_outcome[df_all_outcome[BLOCK_COL].eq(2)]),
        ),
    ])

    for col_idx, (summary_key, title, boundary, traces) in enumerate(panel_specs):
        ax = fig.add_subplot(gs[row_idx, col_idx])
        summaries[f"{prefix}_{summary_key}"] = plot_latency_grid_panel(
            ax=ax,
            traces=traces,
            title=title,
            boundary=boundary,
            n_bins_per_type=n_bins_per_type,
            row_idx=row_idx,
            col_idx=col_idx,
            row_label=row_label,
        )


def make_latency_by_isi_contingency_grid_figure(
    data_path=NEGATIVE_FILTER_DATA_PATH,
    output_path=None,
    n_bins_per_type=N_BINS_PER_TYPE,
    exclude_warmup=True,
    side_data_path=None,
    side_df=None,
):
    """Create a 14-column x 3-row latency-by-ISI figure split by contingency."""
    df = prepare_latency_dataframe(data_path, exclude_warmup=exclude_warmup)

    if side_df is not None or side_data_path is not None:
        side_data = prepare_side_latency_dataframe(
            path=side_data_path,
            source_df=side_df,
            exclude_warmup=exclude_warmup,
        )
    else:
        side_data = add_side_group(df.copy())
        side_data = side_data.dropna(subset=["physical_lick_side", SIDE_GROUP_COL]).copy()

    reverse_df = df[contingency_mask(df, 2, EXPECTED_REVERSE_SUBJECTS)].copy()
    normal_df = df[contingency_mask(df, 1, EXPECTED_NORMAL_SUBJECTS)].copy()
    reverse_side_data = side_data[contingency_mask(side_data, 2, EXPECTED_REVERSE_SUBJECTS)].copy()
    normal_side_data = side_data[contingency_mask(side_data, 1, EXPECTED_NORMAL_SUBJECTS)].copy()

    row_specs = [
        ("all_mice", "All mice", df, side_data),
        ("reverse", "Reverse\nMC06/MC07/MC08", reverse_df, reverse_side_data),
        ("normal", "Normal\nYH24LG/MC16/MC17", normal_df, normal_side_data),
    ]

    nrows = len(row_specs)
    ncols = 14
    fig = plt.figure(figsize=(5.5 * ncols, 5.0 * nrows), constrained_layout=False)
    gs = gridspec.GridSpec(
        nrows=nrows,
        ncols=ncols,
        figure=fig,
        width_ratios=[1] * ncols,
        height_ratios=[1] * nrows,
        wspace=0.12,
        hspace=0.16,
    )

    summaries = {}
    for row_idx, (prefix, row_label, row_df, row_side_data) in enumerate(row_specs):
        add_latency_grid_row(
            fig=fig,
            gs=gs,
            row_idx=row_idx,
            row_label=row_label,
            row_df=row_df,
            row_side_data=row_side_data,
            n_bins_per_type=n_bins_per_type,
            summaries=summaries,
            prefix=prefix,
        )

    fig.subplots_adjust(left=0.025, right=0.995, bottom=0.07, top=0.91, wspace=0.12, hspace=0.16)

    if output_path:
        fig.savefig(output_path, dpi=600, bbox_inches="tight")

    return df, fig, summaries


negative_filtered_df, negative_lick_mask, negative_exclusion_summary, negative_excluded_by_trial_type = (
    load_negative_lick_filtered_dataframe(NEGATIVE_FILTER_DATA_PATH)
)
side_negative_filtered_df, side_negative_lick_mask, side_negative_exclusion_summary, side_negative_excluded_by_trial_type = (
    load_negative_lick_filtered_dataframe(SIDE_NEGATIVE_FILTER_DATA_PATH)
)

print("Negative-lick trial exclusion summary")
display(negative_exclusion_summary)
print("Excluded trials by trial type")
display(negative_excluded_by_trial_type)
print("Side-panel pooled data negative-lick exclusion summary")
display(side_negative_exclusion_summary)
print("Side-panel excluded trials by trial type")
display(side_negative_excluded_by_trial_type)

with read_csv_from_filtered_dataframe(negative_filtered_df):
    df_no_negative_licks, fig_no_negative_licks, stats_df_no_negative_licks = make_licking_analysis_figure(
        data_path=NEGATIVE_FILTER_DATA_PATH,
        output_path=None,
        exclude_warmup=True,
    )
fig_no_negative_licks.suptitle(
    "Licking Summary: Trials With Any Negative Lick Time Excluded",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)
fig_no_negative_licks.savefig(
    NO_NEGATIVE_LICKING_SUMMARY_OUTPUT_PATH,
    dpi=600,
    bbox_inches="tight",
)
print(f"Saved {NO_NEGATIVE_LICKING_SUMMARY_OUTPUT_PATH}")
plt.show()
display(stats_df_no_negative_licks)

contingency_check_summary = summarize_contingency_subjects(negative_filtered_df)
print("Contingency check for latency-by-ISI rows")
display(contingency_check_summary)
verify_expected_contingency_subjects(negative_filtered_df)

with read_csv_from_filtered_dataframe(negative_filtered_df), temporary_globals(
    COLORS=NEGATIVE_FILTER_LATENCY_COLORS,
    style_axis=negative_filter_latency_style_axis,
):
    df_latency_no_negative_licks, fig_latency_no_negative_licks, latency_summaries_no_negative_licks = (
        make_latency_by_isi_contingency_grid_figure(
            data_path=NEGATIVE_FILTER_DATA_PATH,
            output_path=None,
            exclude_warmup=True,
            side_df=side_negative_filtered_df,
        )
    )
fig_latency_no_negative_licks.suptitle(
    "Lick Latency by ISI Bin: Trials With Any Negative Lick Time Excluded",
    fontsize=16,
    fontweight="bold",
)
fig_latency_no_negative_licks.savefig(
    NO_NEGATIVE_LATENCY_ISI_OUTPUT_PATH,
    dpi=600,
    bbox_inches="tight",
)
print(f"Saved {NO_NEGATIVE_LATENCY_ISI_OUTPUT_PATH}")
plt.show()

with read_csv_from_filtered_dataframe(negative_filtered_df), suppress_figure_writes():
    df_raster_no_negative_licks, fig_raster_no_negative_licks = make_licking_raster_figure()
fig_raster_no_negative_licks.suptitle(
    "Mouse Licking Aligned to Spout Movement: Trials With Any Negative Lick Time Excluded",
    fontsize=16,
    fontweight="bold",
)
fig_raster_no_negative_licks.savefig(
    NO_NEGATIVE_RASTER_OUTPUT_PATH,
    dpi=600,
    bbox_inches="tight",
)
print(f"Saved {NO_NEGATIVE_RASTER_OUTPUT_PATH}")
plt.show()

Negative-lick trial exclusion summary


,dataset,trials
0,Original CSV,291
1,Excluded: at least one negative lick time,0
2,Retained for filtered analyses,291


Excluded trials by trial type


,trial_type,excluded_trials


Side-panel pooled data negative-lick exclusion summary


...

,comparison,value_col,test,statistic,p_raw,p_holm,n_trials,n_sessions,mean_diff,effect_dz,normality_p,panel,section,order
0,Short vs Long,all_lick_count,N/A,NaN,NaN,NaN,239,1,NaN,NaN,NaN,col1,COUNTS,10
1,SB Com vs SB Rare,all_lick_count,N/A,NaN,NaN,NaN,120,1,NaN,NaN,NaN,col2,COUNTS,11
2,LB Com vs LB Rare,all_lick_count,N/A,NaN,NaN,NaN,119,1,NaN,NaN,NaN,col2,COUNTS,12
3,NB Short vs NB Long,all_lick_count,N/A,NaN,NaN,NaN,37,1,NaN,NaN,NaN,col2,COUNTS,13
4,Short: Rew vs Pun,all_lick_count,N/A,NaN,NaN,NaN,126,1,NaN,NaN,NaN,col3,COUNTS,14
5,Long: Rew vs Pun,all_lick_count,N/A,NaN,NaN,NaN,113,1,NaN,NaN,NaN,col3,COUNTS,15
6,SB C: Rew vs Pun,all_lick_count,N/A,NaN,NaN,NaN,101,1,NaN,NaN,NaN,col4,COUNTS,16
7,SB R: Rew vs Pun,all_lick_count,N/A,NaN,NaN,NaN,19,0,NaN,NaN,NaN,col4,COUNTS,17
8,LB C: Rew vs Pun,all_lick_count,N/A,NaN,NaN,NaN,94,1,NaN,NaN,NaN,col4,COUNTS,18
9,LB R: Rew vs Pun,all_lick_count,N/A,NaN,NaN,NaN,25,1,NaN,NaN,NaN,col4,COUNTS,19


Contingency check for latency-by-ISI rows


,subject,Contingency_idx,Contingency,sessions
0,MC06,2,Reverse - Long/Left,1


Saved /home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/licking_latency_by_isi_no_negative_lick_trials.pdf


Saved /home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/licking_raster_and_average_trajectory_no_negative_lick_trials.pdf


In [6]:
# Histogram of first positive licking latency.
# Uses the saved scalar latency column and does not modify any files.
import ast
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LATENCY_HIST_DATA_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/all_subjects_final_data.csv"
LATENCY_HIST_COL = "lick_reaction_time"
LATENCY_HIST_SESSION_COL = "session_idx"
NEGATIVE_LICK_TIMES_COL = "all_lick_reaction_times"
LATENCY_HIST_EXCLUDE_WARMUP = True
LATENCY_HIST_BIN_WIDTH = 0.05
LATENCY_HIST_MAX_SECONDS = 3.0
LATENCY_HIST_OUTPUT_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/licking_latency_histogram.pdf"
NEGATIVE_LICK_PIE_OUTPUT_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/negative_lick_summary_pies.pdf"

latency_hist_df = pd.read_csv(LATENCY_HIST_DATA_PATH).copy()
latency_hist_df[LATENCY_HIST_COL] = pd.to_numeric(
    latency_hist_df[LATENCY_HIST_COL], errors="coerce"
)

if LATENCY_HIST_EXCLUDE_WARMUP and "warm_up" in latency_hist_df.columns:
    latency_hist_df = latency_hist_df[latency_hist_df["warm_up"].eq(0)].copy()

latency_hist_df = latency_hist_df[
    latency_hist_df[LATENCY_HIST_COL].notna()
    & latency_hist_df[LATENCY_HIST_COL].ge(0)
].copy()
latency_hist_df["trial_name"] = latency_hist_df["trial_type"].map({1: "Short", 2: "Long"})
latency_hist_df["outcome_strict"] = latency_hist_df["outcome"].replace({
    "RewardNaive": "Reward",
    "PunishNaive": "Punish",
})

hist_bins = np.arange(
    0,
    min(LATENCY_HIST_MAX_SECONDS, latency_hist_df[LATENCY_HIST_COL].max())
    + LATENCY_HIST_BIN_WIDTH,
    LATENCY_HIST_BIN_WIDTH,
)

fig_latency_hist, axes_latency_hist = plt.subplots(1, 4, figsize=(23, 5), sharey=True)

axes_latency_hist[0].hist(
    latency_hist_df[LATENCY_HIST_COL],
    bins=hist_bins,
    color="0.30",
    alpha=0.85,
)
axes_latency_hist[0].set_title("All Valid Trials")
axes_latency_hist[0].set_xlabel("Lick latency (s)")
axes_latency_hist[0].set_ylabel("Trial count")

for label, color in [("Short", "#17becf"), ("Long", "#9467bd")]:
    values = latency_hist_df.loc[
        latency_hist_df["trial_name"].eq(label), LATENCY_HIST_COL
    ]
    axes_latency_hist[1].hist(
        values,
        bins=hist_bins,
        histtype="step",
        linewidth=2,
        color=color,
        label=f"{label} (n={len(values)})",
    )
axes_latency_hist[1].set_title("Short vs Long")
axes_latency_hist[1].set_xlabel("Lick latency (s)")
axes_latency_hist[1].legend(frameon=False)

for label, color in [("Reward", "#1f77b4"), ("Punish", "#d62728")]:
    values = latency_hist_df.loc[
        latency_hist_df["outcome_strict"].eq(label), LATENCY_HIST_COL
    ]
    axes_latency_hist[2].hist(
        values,
        bins=hist_bins,
        histtype="step",
        linewidth=2,
        color=color,
        label=f"{label} (n={len(values)})",
    )
axes_latency_hist[2].set_title("Reward vs Punish")
axes_latency_hist[2].set_xlabel("Lick latency (s)")
axes_latency_hist[2].legend(frameon=False)

for label, color in [("Left", "#0057b8"), ("Right", "#b30000")]:
    values = latency_hist_df.loc[
        latency_hist_df["lick_side"].eq(label), LATENCY_HIST_COL
    ]
    axes_latency_hist[3].hist(
        values,
        bins=hist_bins,
        histtype="step",
        linewidth=2,
        color=color,
        label=f"{label} (n={len(values)})",
    )
axes_latency_hist[3].set_title("Left vs Right")
axes_latency_hist[3].set_xlabel("Lick latency (s)")
axes_latency_hist[3].legend(frameon=False)

for ax in axes_latency_hist:
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlim(0, hist_bins[-1] if len(hist_bins) else LATENCY_HIST_MAX_SECONDS)

fig_latency_hist.suptitle(
    "Histogram of First Positive Licking Latency",
    fontsize=15,
    fontweight="bold",
)
fig_latency_hist.tight_layout()
fig_latency_hist.savefig(
    LATENCY_HIST_OUTPUT_PATH,
    dpi=600,
    bbox_inches="tight",
)
print(f"Saved {LATENCY_HIST_OUTPUT_PATH}")
plt.show()

latency_hist_summary = latency_hist_df[LATENCY_HIST_COL].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]
)


def parse_negative_lick_values(value):
    """Parse one all_lick_reaction_times cell into numeric lick times."""
    if isinstance(value, np.ndarray):
        values = value.tolist()
    elif isinstance(value, (list, tuple)):
        values = list(value)
    elif value is None or (isinstance(value, float) and pd.isna(value)):
        values = []
    elif isinstance(value, str):
        text = value.strip()
        if not text:
            values = []
        else:
            try:
                values = json.loads(text)
            except json.JSONDecodeError:
                try:
                    values = ast.literal_eval(text)
                except (ValueError, SyntaxError, TypeError):
                    values = []
    else:
        values = []

    numeric_values = []
    for item in values if isinstance(values, list) else []:
        try:
            numeric_values.append(float(item))
        except (TypeError, ValueError):
            continue
    return numeric_values


def has_negative_lick_value(value):
    """True when a trial contains at least one pre-WindowChoice lick time."""
    return any(lick_time < 0 for lick_time in parse_negative_lick_values(value))


negative_lick_df = pd.read_csv(LATENCY_HIST_DATA_PATH).copy()
if NEGATIVE_LICK_TIMES_COL not in negative_lick_df.columns:
    raise ValueError(f"Missing required column: {NEGATIVE_LICK_TIMES_COL}")

negative_lick_df["has_negative_lick"] = negative_lick_df[NEGATIVE_LICK_TIMES_COL].apply(
    has_negative_lick_value
)
negative_lick_df["trial_name"] = negative_lick_df["trial_type"].map({1: "Short", 2: "Long"})
negative_trial_df = negative_lick_df[negative_lick_df["has_negative_lick"]].copy()

session_identifier = "session_uid" if "session_uid" in negative_lick_df.columns else LATENCY_HIST_SESSION_COL
problem_sessions = (
    negative_lick_df
    .groupby(["subject", session_identifier], dropna=False)["has_negative_lick"]
    .any()
    .reset_index()
)

negative_trials_by_subject = negative_trial_df["subject"].value_counts().sort_index()
problem_sessions_by_subject = (
    problem_sessions[problem_sessions["has_negative_lick"]]
    ["subject"]
    .value_counts()
    .sort_index()
)
negative_trials_by_trial_type = (
    negative_trial_df["trial_name"]
    .value_counts()
    .reindex(["Short", "Long"])
    .fillna(0)
    .astype(int)
)
all_session_problem_counts = pd.Series({
    "With negative licks": int(problem_sessions["has_negative_lick"].sum()),
    "Without negative licks": int((~problem_sessions["has_negative_lick"]).sum()),
})


def plot_count_pie(ax, counts, title):
    """Draw a pie chart with count-aware labels."""
    counts = counts[counts > 0]
    if counts.empty:
        ax.text(0.5, 0.5, "No data", transform=ax.transAxes, ha="center", va="center")
        ax.set_title(title)
        ax.axis("off")
        return

    labels = [f"{label}\n{int(value):,}" for label, value in counts.items()]
    ax.pie(
        counts.values,
        labels=labels,
        autopct="%1.1f%%",
        startangle=90,
        counterclock=False,
        textprops={"fontsize": 10},
    )
    ax.set_title(title, fontweight="bold")


fig_negative_lick_pies, axes_negative_lick_pies = plt.subplots(2, 2, figsize=(15, 12))
plot_count_pie(
    axes_negative_lick_pies[0, 0],
    negative_trials_by_subject,
    "Negative-Lick Trials by Subject",
)
plot_count_pie(
    axes_negative_lick_pies[0, 1],
    problem_sessions_by_subject,
    "Sessions With Negative Licks by Subject",
)
plot_count_pie(
    axes_negative_lick_pies[1, 0],
    all_session_problem_counts,
    "All Sessions: Negative-Lick Problem",
)
plot_count_pie(
    axes_negative_lick_pies[1, 1],
    negative_trials_by_trial_type,
    "Negative-Lick Trials: Short vs Long",
)
fig_negative_lick_pies.suptitle(
    "Negative Lick Summary",
    fontsize=16,
    fontweight="bold",
)
fig_negative_lick_pies.tight_layout()
fig_negative_lick_pies.savefig(
    NEGATIVE_LICK_PIE_OUTPUT_PATH,
    dpi=600,
    bbox_inches="tight",
)
print(f"Saved {NEGATIVE_LICK_PIE_OUTPUT_PATH}")
plt.show()

negative_lick_subject_summary = pd.concat(
    [
        negative_trials_by_subject.rename("negative_trials"),
        problem_sessions_by_subject.rename("problem_sessions"),
    ],
    axis=1,
).fillna(0).astype(int)
negative_lick_trial_type_summary = negative_trials_by_trial_type.rename("negative_trials")

print("Negative lick summary by subject")
display(negative_lick_subject_summary)
print("All-session problem counts")
display(all_session_problem_counts.rename("sessions").to_frame())
print("Negative lick trials by trial type")
display(negative_lick_trial_type_summary.to_frame())

latency_hist_summary


Saved /home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/licking_latency_histogram.pdf


Saved /home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/negative_lick_summary_pies.pdf
Negative lick summary by subject


,negative_trials,problem_sessions
subject,,


All-session problem counts


,sessions
With negative licks,0
Without negative licks,1


Negative lick trials by trial type


,negative_trials
trial_name,
Short,0
Long,0


count    276.000000
mean       0.252487
std        0.138669
min        0.055700
25%        0.158050
50%        0.200850
75%        0.311250
90%        0.435250
95%        0.527050
max        0.890700
Name: lick_reaction_time, dtype: float64

In [7]:
# Non-negative lick-latency summary grid by choice, ISI, and outcome
import ast
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy import stats

LATENCY_GRID_DATA_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/all_subjects_final_data.csv"
LATENCY_GRID_OUTPUT_PATH = "/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/latency_choice_isi_outcome_nonnegative_grid.pdf"
LATENCY_COL = "lick_reaction_time"
LICK_DETAILS_COL = "all_lick_details"
NEGATIVE_LICK_ARRAY_COLUMNS = ["all_lick_reaction_times", "all_lick_start_times_from_choice"]

REWARD_OUTCOMES = {"Reward", "RewardNaive"}
PUNISH_OUTCOMES = {"Punish", "PunishNaive"}
CHOICE_SIDES = ["Left", "Right"]
TRIAL_NAMES = {1: "Short", 2: "Long"}

CHOICE_COLORS = {"Left": "#0057b8", "Right": "#b30000"}
TRIAL_COLORS = {"Short": "#17becf", "Long": "#9467bd"}
OUTCOME_COLORS = {"Correct": "#1f77b4", "Incorrect": "#d62728"}


def latency_grid_parse_list_cell(value):
    """Parse JSON/Python-list cells from the CSV."""
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            try:
                parsed = ast.literal_eval(text)
            except (SyntaxError, ValueError, TypeError):
                return []
        return parsed if isinstance(parsed, list) else []
    return []


def latency_grid_numeric_values(value):
    values = []
    for item in latency_grid_parse_list_cell(value):
        try:
            numeric = float(item)
        except (TypeError, ValueError):
            continue
        if np.isfinite(numeric):
            values.append(numeric)
    return values


def latency_grid_has_negative_lick(value):
    return any(lick_time < 0 for lick_time in latency_grid_numeric_values(value))


def latency_grid_first_post_choice_side(value):
    """Return physical side of the first lick at or after WindowChoice."""
    for lick in latency_grid_parse_list_cell(value):
        if not isinstance(lick, dict):
            continue
        try:
            reaction_time = float(lick.get("reaction_time", lick.get("start_time_from_choice", np.nan)))
        except (TypeError, ValueError):
            continue
        if not np.isfinite(reaction_time) or reaction_time < 0:
            continue
        side = lick.get("side")
        if side in CHOICE_SIDES:
            return side
    return np.nan


def latency_grid_outcome_class(outcome):
    if outcome in REWARD_OUTCOMES:
        return "Correct"
    if outcome in PUNISH_OUTCOMES:
        return "Incorrect"
    return np.nan


def prepare_nonnegative_latency_grid_dataframe(path=LATENCY_GRID_DATA_PATH):
    """Load, filter, and label trials for the requested latency analysis."""
    required = {
        "subject", "trial_type", "block_type", "outcome", LATENCY_COL,
        "lick_side", LICK_DETAILS_COL,
    }
    df = pd.read_csv(path).copy()
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    for column in [LATENCY_COL, "trial_type", "block_type", "Contingency_idx", "warm_up"]:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

    if "warm_up" in df.columns:
        df = df[df["warm_up"].eq(0)].copy()

    negative_mask = pd.Series(False, index=df.index)
    missing_negative_columns = []
    for column in NEGATIVE_LICK_ARRAY_COLUMNS:
        if column in df.columns:
            negative_mask = negative_mask | df[column].apply(latency_grid_has_negative_lick)
        else:
            missing_negative_columns.append(column)

    df = df.loc[~negative_mask].copy()
    df = df[df[LATENCY_COL].notna() & df[LATENCY_COL].ge(0)].copy()

    df["choice_side"] = df[LICK_DETAILS_COL].apply(latency_grid_first_post_choice_side)
    df["choice_side"] = df["choice_side"].fillna(df["lick_side"])
    df.loc[~df["choice_side"].isin(CHOICE_SIDES), "choice_side"] = np.nan

    df["outcome_class"] = df["outcome"].apply(latency_grid_outcome_class)
    df["trial_name"] = df["trial_type"].map(TRIAL_NAMES)

    df["is_common"] = np.nan
    df.loc[(df["block_type"] == 1) & (df["trial_type"] == 1), "is_common"] = 1
    df.loc[(df["block_type"] == 1) & (df["trial_type"] == 2), "is_common"] = 0
    df.loc[(df["block_type"] == 2) & (df["trial_type"] == 2), "is_common"] = 1
    df.loc[(df["block_type"] == 2) & (df["trial_type"] == 1), "is_common"] = 0
    df["rarity_label"] = df["is_common"].map({1.0: "Common", 0.0: "Rare"})

    if "session_uid" in df.columns:
        df["latency_grid_session"] = df["session_uid"].astype(str)
    elif {"subject", "session_idx"}.issubset(df.columns):
        df["latency_grid_session"] = df["subject"].astype(str) + "_S" + df["session_idx"].astype(str)
    elif "session_idx" in df.columns:
        df["latency_grid_session"] = df["session_idx"].astype(str)
    else:
        df["latency_grid_session"] = "session_1"

    df.attrs["missing_negative_columns"] = missing_negative_columns
    return df


def latency_grid_counts(data):
    return {
        "subjects": int(data["subject"].nunique()) if "subject" in data.columns else 0,
        "sessions": int(data["latency_grid_session"].nunique()) if "latency_grid_session" in data.columns else 0,
        "trials": int(len(data)),
    }


def latency_grid_condition_summary(data, mask):
    subset = data.loc[mask].dropna(subset=[LATENCY_COL, "latency_grid_session"])
    session_means = subset.groupby("latency_grid_session", observed=True)[LATENCY_COL].mean()
    if session_means.empty:
        return np.nan, np.nan, 0, 0, 0
    mean = float(session_means.mean())
    sem = float(session_means.sem()) if len(session_means) > 1 else 0.0
    n_subjects = int(subset["subject"].nunique()) if "subject" in subset.columns else 0
    return mean, sem, int(len(session_means)), n_subjects, int(len(subset))


def latency_grid_p_to_stars(p_value):
    if pd.isna(p_value):
        return "n/a"
    if p_value < 0.0001:
        return "****"
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return "ns"


def latency_grid_paired_session_test(data, mask_a, mask_b):
    a = (
        data.loc[mask_a]
        .dropna(subset=[LATENCY_COL, "latency_grid_session"])
        .groupby("latency_grid_session", observed=True)[LATENCY_COL]
        .mean()
        .rename("a")
    )
    b = (
        data.loc[mask_b]
        .dropna(subset=[LATENCY_COL, "latency_grid_session"])
        .groupby("latency_grid_session", observed=True)[LATENCY_COL]
        .mean()
        .rename("b")
    )
    paired = pd.concat([a, b], axis=1, join="inner").dropna()
    if len(paired) < 2:
        return {"test": "insufficient paired sessions", "p": np.nan, "n_pairs": int(len(paired))}

    differences = paired["a"] - paired["b"]
    if np.allclose(differences, 0):
        return {"test": "all paired differences zero", "p": 1.0, "n_pairs": int(len(paired))}

    shapiro_p = np.nan
    if len(paired) >= 3 and differences.nunique() > 1:
        try:
            shapiro_p = float(stats.shapiro(differences).pvalue)
        except (ValueError, FloatingPointError):
            shapiro_p = np.nan

    if len(paired) >= 3 and pd.notna(shapiro_p) and shapiro_p >= 0.05:
        test_result = stats.ttest_rel(paired["a"], paired["b"], nan_policy="omit")
        test_name = "paired t-test"
    else:
        try:
            test_result = stats.wilcoxon(paired["a"], paired["b"], zero_method="wilcox")
            test_name = "Wilcoxon signed-rank"
        except ValueError:
            test_result = stats.ttest_rel(paired["a"], paired["b"], nan_policy="omit")
            test_name = "paired t-test fallback"

    return {
        "test": test_name,
        "p": float(test_result.pvalue),
        "n_pairs": int(len(paired)),
        "shapiro_p": shapiro_p,
    }


def latency_grid_add_significance_bar(ax, x1, x2, y, height, label):
    ax.plot(
        [x1, x1, x2, x2],
        [y, y + height, y + height, y],
        color="0.15",
        linewidth=0.9,
        clip_on=False,
    )
    ax.text(
        (x1 + x2) / 2,
        y + height,
        label,
        ha="center",
        va="bottom",
        fontsize=7,
        color="0.15",
        clip_on=False,
    )


def latency_grid_style_axis(ax):
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_axisbelow(True)
    ax.relim()
    ax.autoscale_view(scalex=False, scaley=True)
    ax.margins(y=0.18)
    ax.tick_params(axis="x", labelsize=8)
    ax.tick_params(axis="y", labelsize=8)


def latency_grid_point_label(label, n_subjects, n_trials):
    return f"{label}: nSub={n_subjects}, nTr={n_trials}"


def plot_latency_grouped_points(ax, data, grouped_specs, title, ylabel=None):
    centers = np.arange(len(grouped_specs), dtype=float)
    legend_handles = []
    pair_annotations = []

    for group_index, (_, specs) in enumerate(grouped_specs):
        offsets = np.linspace(-0.08, 0.08, len(specs)) if len(specs) > 1 else np.array([0.0])
        plotted_points = []
        for offset, spec in zip(offsets, specs):
            mean, sem, _, n_subjects, n_trials = latency_grid_condition_summary(data, spec["mask"])
            x_position = centers[group_index] + offset
            legend_label = latency_grid_point_label(spec["label"], n_subjects, n_trials)
            legend_handles.append(
                Line2D(
                    [0], [0], marker="o", color=spec["color"], linestyle="None",
                    markersize=5.5, label=legend_label,
                )
            )
            point = {"x": x_position, "mean": mean, "sem": sem, "spec": spec}
            plotted_points.append(point)
            if pd.isna(mean):
                continue
            ax.errorbar(
                x_position,
                mean,
                yerr=sem,
                fmt="o",
                color=spec["color"],
                ecolor=spec["color"],
                markersize=5.5,
                capsize=3,
                elinewidth=1.2,
                markeredgecolor="white",
                markeredgewidth=0.5,
                zorder=3,
            )

        if len(specs) == 2:
            pair_annotations.append(
                (
                    plotted_points,
                    latency_grid_paired_session_test(data, specs[0]["mask"], specs[1]["mask"]),
                )
            )

    y_min, y_max = ax.get_ylim()
    y_range = max(y_max - y_min, 0.05)
    bar_height = y_range * 0.025
    bar_offset = y_range * 0.06

    for plotted_points, test_result in pair_annotations:
        finite_points = [point for point in plotted_points if pd.notna(point["mean"])]
        if len(finite_points) != 2:
            continue
        y_top = max(point["mean"] + (0 if pd.isna(point["sem"]) else point["sem"]) for point in finite_points)
        latency_grid_add_significance_bar(
            ax,
            finite_points[0]["x"],
            finite_points[1]["x"],
            y_top + bar_offset,
            bar_height,
            latency_grid_p_to_stars(test_result["p"]),
        )

    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xticks(centers)
    ax.set_xticklabels([group_label for group_label, _ in grouped_specs], rotation=0)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=9, fontweight="bold")
    else:
        ax.set_ylabel("")
    latency_grid_style_axis(ax)
    if legend_handles:
        ax.legend(
            handles=legend_handles,
            loc="center left",
            bbox_to_anchor=(1.02, 0.5),
            frameon=False,
            fontsize=5.8,
            handlelength=0.9,
            handletextpad=0.35,
            labelspacing=0.35,
            borderaxespad=0.0,
        )


def plot_latency_vs_choice(ax, data, row_label):
    data = data[data["choice_side"].isin(CHOICE_SIDES)].copy()
    grouped_specs = [
        (
            "All\ntrials",
            [
                {"mask": data["choice_side"].eq("Left"), "color": CHOICE_COLORS["Left"], "label": "All Left"},
                {"mask": data["choice_side"].eq("Right"), "color": CHOICE_COLORS["Right"], "label": "All Right"},
            ],
        ),
        (
            "Correct\ntrials",
            [
                {"mask": data["outcome_class"].eq("Correct") & data["choice_side"].eq("Left"), "color": CHOICE_COLORS["Left"], "label": "Correct Left"},
                {"mask": data["outcome_class"].eq("Correct") & data["choice_side"].eq("Right"), "color": CHOICE_COLORS["Right"], "label": "Correct Right"},
            ],
        ),
        (
            "Incorrect\ntrials",
            [
                {"mask": data["outcome_class"].eq("Incorrect") & data["choice_side"].eq("Left"), "color": CHOICE_COLORS["Left"], "label": "Incorrect Left"},
                {"mask": data["outcome_class"].eq("Incorrect") & data["choice_side"].eq("Right"), "color": CHOICE_COLORS["Right"], "label": "Incorrect Right"},
            ],
        ),
    ]
    plot_latency_grouped_points(ax, data, grouped_specs, "Latency vs Choice", ylabel=f"{row_label}\nLatency (s)")
    return data


def plot_latency_vs_isi(ax, data, is_common, title):
    data = data[
        data["is_common"].eq(is_common)
        & data["outcome_class"].isin(["Correct", "Incorrect"])
        & data["trial_name"].isin(["Short", "Long"])
    ].copy()
    grouped_specs = []
    for outcome_class in ["Correct", "Incorrect"]:
        grouped_specs.append(
            (
                outcome_class,
                [
                    {
                        "mask": data["outcome_class"].eq(outcome_class) & data["trial_name"].eq("Short"),
                        "color": TRIAL_COLORS["Short"],
                        "label": f"{outcome_class} Short",
                    },
                    {
                        "mask": data["outcome_class"].eq(outcome_class) & data["trial_name"].eq("Long"),
                        "color": TRIAL_COLORS["Long"],
                        "label": f"{outcome_class} Long",
                    },
                ],
            )
        )
    plot_latency_grouped_points(ax, data, grouped_specs, title)
    return data


def plot_latency_vs_outcome(ax, data):
    data = data[
        data["is_common"].isin([0, 1])
        & data["outcome_class"].isin(["Correct", "Incorrect"])
        & data["trial_name"].isin(["Short", "Long"])
    ].copy()
    grouped_specs = []
    for rarity_value, rarity_label in [(1, "Common"), (0, "Rare")]:
        for trial_name in ["Short", "Long"]:
            grouped_specs.append(
                (
                    f"{rarity_label}\n{trial_name}",
                    [
                        {
                            "mask": data["is_common"].eq(rarity_value)
                            & data["trial_name"].eq(trial_name)
                            & data["outcome_class"].eq("Correct"),
                            "color": OUTCOME_COLORS["Correct"],
                            "label": f"{rarity_label} {trial_name} Correct",
                        },
                        {
                            "mask": data["is_common"].eq(rarity_value)
                            & data["trial_name"].eq(trial_name)
                            & data["outcome_class"].eq("Incorrect"),
                            "color": OUTCOME_COLORS["Incorrect"],
                            "label": f"{rarity_label} {trial_name} Incorrect",
                        },
                    ],
                )
            )
    plot_latency_grouped_points(ax, data, grouped_specs, "Latency vs Outcome")
    return data


def latency_grid_row_specs(df):
    specs = [("All subjects", df.copy())]
    if "Contingency_idx" in df.columns:
        specs.append(("Reverse\ncontingency", df[df["Contingency_idx"].eq(2)].copy()))
        specs.append(("Normal\ncontingency", df[df["Contingency_idx"].eq(1)].copy()))
    elif "Contingency" in df.columns:
        contingency_text = df["Contingency"].astype(str).str.lower()
        specs.append(("Reverse\ncontingency", df[contingency_text.str.contains("reverse", na=False)].copy()))
        specs.append(("Normal\ncontingency", df[contingency_text.str.contains("normal", na=False)].copy()))

    for subject in pd.unique(df["subject"].dropna().astype(str)):
        specs.append((subject, df[df["subject"].astype(str).eq(subject)].copy()))
    return specs


def make_nonnegative_latency_grid_figure(data_path=LATENCY_GRID_DATA_PATH, output_path=LATENCY_GRID_OUTPUT_PATH):
    df = prepare_nonnegative_latency_grid_dataframe(data_path)
    row_specs = latency_grid_row_specs(df)
    nrows = len(row_specs)
    ncols = 4

    fig = plt.figure(figsize=(30, max(18, 2.9 * nrows)))
    gs = gridspec.GridSpec(
        nrows=nrows,
        ncols=ncols,
        figure=fig,
        left=0.07,
        right=0.93,
        bottom=0.05,
        top=0.90,
        hspace=0.62,
        wspace=1.35,
    )

    column_titles = [
        "Choice side: All/Correct/Incorrect",
        "Common trials: Correct/Incorrect x Short/Long",
        "Rare trials: Correct/Incorrect x Short/Long",
        "Common/Rare x Short/Long x Outcome",
    ]
    panel_summary_rows = []

    for row_index, (row_label, row_data) in enumerate(row_specs):
        axes = [fig.add_subplot(gs[row_index, col]) for col in range(ncols)]
        if row_index == 0:
            for ax, col_title in zip(axes, column_titles):
                ax.set_title(col_title, fontsize=11, fontweight="bold")

        panel_data = plot_latency_vs_choice(axes[0], row_data, row_label)
        panel_summary_rows.append({"row": row_label.replace("\n", " "), "panel": "Latency vs Choice", **latency_grid_counts(panel_data)})

        panel_data = plot_latency_vs_isi(axes[1], row_data, 1, "Common Trials")
        panel_summary_rows.append({"row": row_label.replace("\n", " "), "panel": "Common: Latency vs ISI", **latency_grid_counts(panel_data)})

        panel_data = plot_latency_vs_isi(axes[2], row_data, 0, "Rare Trials")
        panel_summary_rows.append({"row": row_label.replace("\n", " "), "panel": "Rare: Latency vs ISI", **latency_grid_counts(panel_data)})

        panel_data = plot_latency_vs_outcome(axes[3], row_data)
        panel_summary_rows.append({"row": row_label.replace("\n", " "), "panel": "Latency vs Outcome", **latency_grid_counts(panel_data)})

    total_counts = latency_grid_counts(df)
    missing_negative_columns = df.attrs.get("missing_negative_columns", [])
    missing_note = "" if not missing_negative_columns else f" Missing negative-check columns: {missing_negative_columns}."
    fig.suptitle(
        "Lick Latency Summary Using Non-Negative Trials\n"
        f"Warm-up excluded; trials with any negative all-lick time excluded; "
        f"nSub={total_counts['subjects']}, nSess={total_counts['sessions']}, nTr={total_counts['trials']}."
        f"{missing_note}",
        fontsize=14,
        fontweight="bold",
    )

    if output_path:
        fig.savefig(output_path, dpi=600, bbox_inches="tight")
        print(f"Saved {output_path}")

    return df, fig, pd.DataFrame(panel_summary_rows)


latency_grid_df, latency_grid_fig, latency_grid_summary = make_nonnegative_latency_grid_figure()
plt.show()
display(latency_grid_summary)

Saved /home/ihsan/Desktop/data/Georgia_Tech/2AFC/Codes/Model_behavior/Bayesian_model_v2/results_raw_behavior_by_winner/M3/latency_choice_isi_outcome_nonnegative_grid.pdf


,row,panel,subjects,sessions,trials
0,All subjects,Latency vs Choice,1,1,276
1,All subjects,Common: Latency vs ISI,1,1,195
2,All subjects,Rare: Latency vs ISI,1,1,44
3,All subjects,Latency vs Outcome,1,1,239
4,Reverse contingency,Latency vs Choice,1,1,276
5,Reverse contingency,Common: Latency vs ISI,1,1,195
6,Reverse contingency,Rare: Latency vs ISI,1,1,44
7,Reverse contingency,Latency vs Outcome,1,1,239
8,Normal contingency,Latency vs Choice,0,0,0
9,Normal contingency,Common: Latency vs ISI,0,0,0
